In [ ]:
import pandas as pd
import numpy as np
import json
import os
import math
import calendar
from datetime import datetime, date
from collections import defaultdict

# =============================================================
#  Smart APS HZ V1  —  Horizontal Section Scheduler
# =============================================================
# ADAPTED FROM VT V16. KEY DIFFERENCES:
#   1. NO MULTI-MACHINE PER PART: A part is locked to ONE machine
#      for the entire day, regardless of tools_available. The tools
#      column is read but never used to justify a second machine.
#   2. MAX CHANGEOVERS = 45 (vs 25 in VT).
#   3. FAIR CAPACITY DISTRIBUTION: Before assigning hours, a
#      pre-distribution pass spreads parts evenly across machines
#      so no machine hogs capacity while another starves.
#   4. All sheet/variable names use HZ prefix (HZ_Matrix, etc.)
#   5. Demand-first, then indent, then coverage-days urgency.
# =============================================================

# =============================================================
# SECTION 1 — DAILY SETTINGS
# =============================================================

PLANNING_DATE = date(2026, 4, 9)
INDENT_MONTH  = date(2026, 4, 1)

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS          = 22
AVAILABLE_HOURS_EXTENDED = 23
MIN_RUN_HOURS            = 4
MACHINE_STATE_FILE       = "machine_state_hz.json"

MIN_DAILY_INDENT         = 150
MIN_INDENT_HOURS         = 4.0

SAFETY_DAYS              = 3
TARGET_DAYS              = 5

OPD_SCENARIO_0 = 3.0
OPD_SCENARIO_1 = 3.0
OPD_SCENARIO_2 = 4.0
OPD_SCENARIO_3 = 5.0

W_URGENCY  = 0.55
W_CATEGORY = 0.25
W_INDENT   = 0.20

UTIL_TARGET_PCT  = 90.0
COLOR_PURGE_HRS  = 10 / 60.0

RUNNER_PRIORITY_DAYS = 2.0

MAX_PARTS_PER_MACHINE    = 3
_dynamic_max_parts       = MAX_PARTS_PER_MACHINE

MAX_DAILY_CO             = 45

FORWARD_LOOK_DAYS        = 7

TERMINAL_RELAXATION_PCT  = 0.15

STRATEGIC_BUFFER_DAYS        = 7
STRATEGIC_PRIORITY_DISCOUNT  = 0.5
EFFICIENCY_MODE_UTIL_FLOOR   = 95.0
ABSOLUTE_MAX_DAYS            = 15

# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================

book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
terminal_path   = "C:/Users/Ex0164/Important codes/terminals and raw marterial - vt.xlsx"
demand_path     = "C:/Users/Ex0164/Important codes/Child_for_8TH may_actual.xlsx"
output_path     = f"Smart_APS_HZ_V1_Plan_{PLANNING_DATE.strftime('%Y%m%d')}.xlsx"

MAIN_SHEET_NAME        = "HZ"
MATRIX_SHEET_NAME      = "HZ_Matrix"
MACHINE_COUNT_SHEET    = "HZ_Machine_Part_Count"
CHANGEOVER_SHEET       = "HZ_Changeover"
FIXED_SHEET            = "HZ_Fixed"
TERMINAL_SHEET         = "HZ_Terminals"
TERMINAL_INV_SHEET     = "HZ_Terminal_Inventory"
DEMAND_SHEET_NAME      = "HZ_Demand"
DEMAND_PART_COL        = "Part"
DEMAND_DAILY_COL       = "Daily_Demand"

# =============================================================
# SECTION 4 — WORKING DAYS
# =============================================================

def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(
        1 for d in range(1, total + 1)
        if date(year, month, d).weekday() == 6
    )
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print("\n" + "─"*65)
print(f"  Smart APS HZ V1  —  Horizontal Section Scheduler")
print(f"  Planning date    : {PLANNING_DATE}")
print(f"  Indent month     : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days     : {WORKING_DAYS}  ({TOTAL_DAYS} days - {SUNDAY_COUNT} Sundays)")
print(f"  Safety floor     : {SAFETY_DAYS}d  |  Target ceiling : {TARGET_DAYS}d")
print(f"  Max parts/machine: {MAX_PARTS_PER_MACHINE}")
print(f"  Runner priority  : inv < {RUNNER_PRIORITY_DAYS}x daily")
print(f"  Max daily CO     : {MAX_DAILY_CO}  (HZ = 45)")
print(f"  Forward look     : {FORWARD_LOOK_DAYS} days")
print(f"  Terminal relax   : {int(TERMINAL_RELAXATION_PCT*100)}%  (min-run NON-NEGOTIABLE)")
print(f"  HZ RULE          : ONE machine per part — NO parallel tooling")
print(f"{'='*65}\n")

# =============================================================
# SECTION 5 — LOAD DATA
# =============================================================

print("Loading data...")
hz_parts_raw         = pd.read_excel(book_path,       sheet_name=MAIN_SHEET_NAME)
hz_matrix            = pd.read_excel(matrix_path,     sheet_name=MATRIX_SHEET_NAME)
hz_co_raw            = pd.read_excel(changeover_path, sheet_name=CHANGEOVER_SHEET)
hz_machine_count_raw = pd.read_excel(matrix_path,     sheet_name=MACHINE_COUNT_SHEET)

try:
    hz_fixed_raw = pd.read_excel(matrix_path, sheet_name=FIXED_SHEET)
    print(f"  HZ_Fixed sheet loaded  ({len(hz_fixed_raw)} rows)")
except Exception as _fe:
    hz_fixed_raw = None
    print(f"  WARNING: HZ_Fixed sheet not found ({_fe})")

try:
    hz_terminals_raw      = pd.read_excel(terminal_path, sheet_name=TERMINAL_SHEET)
    hz_terminal_avail_raw = pd.read_excel(terminal_path, sheet_name=TERMINAL_INV_SHEET)
    print(f"  Terminal data loaded from  : {terminal_path}")
except FileNotFoundError:
    hz_terminals_raw      = None
    hz_terminal_avail_raw = None
    print(f"  WARNING: terminal_path not found — terminal constraint DISABLED.")
except Exception as _te:
    hz_terminals_raw      = None
    hz_terminal_avail_raw = None
    print(f"  WARNING: Could not load terminal data ({_te}) — constraint DISABLED.")

# ── Load demand file ──────────────────────────────────────────
demand_monthly_raw = {}
demand_daily_raw   = {}

try:
    demand_df = pd.read_excel(demand_path, sheet_name=DEMAND_SHEET_NAME)
    demand_df.columns = [str(c).strip().lstrip('\ufeff') for c in demand_df.columns]
    print(f"  Demand file columns found  : {list(demand_df.columns)}")
    print(f"  Demand file rows           : {len(demand_df)}")

    _part_col_d = next(
        (c for c in demand_df.columns if c.strip().lower() == DEMAND_PART_COL.lower()), None
    )
    _dem_col_d = next(
        (c for c in demand_df.columns if c.strip().lower() == DEMAND_DAILY_COL.lower()), None
    )
    if _part_col_d is None:
        _part_col_d = next((c for c in demand_df.columns if 'part' in c.strip().lower()), None)
    if _dem_col_d is None:
        _dem_col_d = next(
            (c for c in demand_df.columns if 'daily' in c.strip().lower() or 'demand' in c.strip().lower()), None
        )

    if _part_col_d and _dem_col_d:
        loaded_count = skipped_count = 0
        for _, row in demand_df.iterrows():
            p = row[_part_col_d]
            v = row[_dem_col_d]
            if pd.isna(p) or str(p).strip() == "":
                skipped_count += 1
                continue
            part_key = str(p).strip()
            try:
                daily_dem = float(v) if pd.notna(v) else 0.0
            except (ValueError, TypeError):
                daily_dem = 0.0
            demand_daily_raw[part_key]   = round(daily_dem, 4)
            demand_monthly_raw[part_key] = round(daily_dem * WORKING_DAYS, 4)
            loaded_count += 1
        print(f"  Demand loaded (DAILY)      : {loaded_count} parts")
    else:
        print(f"  WARNING: Demand columns not found — demand constraint DISABLED.")
except FileNotFoundError:
    print(f"  WARNING: demand_path not found — demand constraint DISABLED.")
except Exception as _de:
    print(f"  WARNING: Could not load demand data ({_de}) — demand constraint DISABLED.")

# =============================================================
# SECTION 5A — EFFECTIVE DAILY / MONTHLY HELPERS
# =============================================================

def effective_daily(part: str) -> float:
    ind = indent_daily.get(part, 0.0)
    dem = demand_daily_raw.get(part, 0.0)
    return max(ind, dem)

def effective_monthly(part: str) -> float:
    ind = indent_monthly.get(part, 0.0)
    dem = demand_monthly_raw.get(part, 0.0)
    return max(ind, dem)

def demand_driver(part: str) -> str:
    ind = indent_daily.get(part, 0.0)
    dem = demand_daily_raw.get(part, 0.0)
    if dem > ind + 0.001:
        return f"DEMAND({dem:.2f}>indent {ind:.2f})"
    elif ind > dem + 0.001:
        return f"INDENT({ind:.2f})"
    elif ind > 0:
        return f"EQUAL({ind:.2f})"
    else:
        return "NO_DRIVER"

def is_demand_met(part: str, inv_before: float, produced: float) -> bool:
    dem = demand_daily_raw.get(part, 0.0)
    if dem <= 0:
        return True
    return (inv_before + produced) >= (dem - 0.5)

# =============================================================
# SECTION 6 — PARSE HZ SHEET
# =============================================================

def find_col(df, name, sheet):
    match = next((c for c in df.columns if str(c).strip().lower() == name.lower()), None)
    if match is None:
        raise ValueError(f"Column '{name}' not found in sheet '{sheet}'.\nAvailable: {list(df.columns)}")
    return match

hz_col_part      = find_col(hz_parts_raw, "Part",       MAIN_SHEET_NAME)
hz_col_cycletime = find_col(hz_parts_raw, "Cycle time", MAIN_SHEET_NAME)
hz_col_cavity    = find_col(hz_parts_raw, "Cavity",     MAIN_SHEET_NAME)
hz_col_inventory = find_col(hz_parts_raw, "Inventory",  MAIN_SHEET_NAME)
hz_col_indent    = find_col(hz_parts_raw, "Indent",     MAIN_SHEET_NAME)
hz_col_tools     = find_col(hz_parts_raw, "Tools",      MAIN_SHEET_NAME)
hz_col_color     = find_col(hz_parts_raw, "Color",      MAIN_SHEET_NAME)

data = hz_parts_raw[
    hz_parts_raw[hz_col_part].notna() &
    (hz_parts_raw[hz_col_part].astype(str).str.strip() != "")
].copy()
data = data.drop_duplicates(subset=hz_col_part).copy()
data["Material"] = data[hz_col_part].astype(str).str.strip()
data["_ct"] = pd.to_numeric(data[hz_col_cycletime], errors="coerce").replace(0, np.nan)
data["Rate"] = 3600 / data["_ct"]
data_valid     = data[data["Rate"].notna()].copy()
data_zero_rate = data[data["Rate"].isna()].copy()
print(f"  HZ parts in sheet       : {len(data)}  |  With valid rate: {len(data_valid)}")

# =============================================================
# SECTION 7 — LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        str(k).strip(): (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
        if pd.notna(k) and str(k).strip() != ""
    }

inventory      = safe_dict(data,       "Material", hz_col_inventory)
rate           = safe_dict(data_valid, "Material", "Rate")
indent_monthly = safe_dict(data,       "Material", hz_col_indent)
indent_daily   = {p: round(qty / WORKING_DAYS, 4) for p, qty in indent_monthly.items()}

tools_available = {}
part_color      = {}

for _, row in data.iterrows():
    p = str(row["Material"]).strip()
    v = row[hz_col_tools]
    tools_available[p] = max(1, int(float(v))) if pd.notna(v) and str(v).strip() != "" else 1
    c = row[hz_col_color]
    part_color[p] = str(c).strip().upper() if pd.notna(c) and str(c).strip() not in ("", "nan") else "UNKNOWN"

ALL_KNOWN_COLORS = dict(part_color)

color_groups = {}
for p, c in part_color.items():
    color_groups.setdefault(c, []).append(p)
print(f"  Distinct colours        : {len(color_groups)}")

today_target_qty = {
    p: max(0.0, effective_daily(p) - inventory.get(p, 0.0))
    for p in set(list(indent_monthly.keys()) + list(demand_daily_raw.keys()))
}

# =============================================================
# SECTION 7A — FIXED MACHINE CONSTRAINT
# =============================================================

def build_fixed_machine_dicts(df):
    pfm, mfp = {}, {}
    if df is None or df.empty:
        return pfm, mfp
    machine_col = next((c for c in df.columns if str(c).strip().lower() == "machine"), None)
    if machine_col is None:
        print("  WARNING: HZ_Fixed has no 'Machine' column — fixed constraint disabled")
        return pfm, mfp
    part_cols = [c for c in df.columns if str(c).strip().lower() != "machine"]
    for _, row in df.iterrows():
        machine = row[machine_col]
        if pd.isna(machine) or str(machine).strip() == "":
            continue
        m = str(machine).strip()
        for col in part_cols:
            val = row[col]
            if pd.isna(val) or str(val).strip() in ("", "nan"):
                continue
            p = str(val).strip()
            if p in pfm:
                print(f"  WARNING: Part '{p}' in HZ_Fixed more than once — keeping {pfm[p]}")
                continue
            pfm[p] = m
            mfp.setdefault(m, []).append(p)
    return pfm, mfp

part_fixed_machine, machine_fixed_parts = build_fixed_machine_dicts(hz_fixed_raw)
print(f"  Fixed machine mappings  : {len(part_fixed_machine)} parts")

# =============================================================
# SECTION 7A2 — FIXED MACHINE PHASE HELPERS
# =============================================================

def fixed_machine_phase(machine, current_inventory):
    for p in machine_fixed_parts.get(machine, []):
        daily = effective_daily(p)
        if daily > 0 and current_inventory.get(p, 0) < SAFETY_DAYS * daily:
            return "A"
    return "B"

def pick_fixed_part_for_today(machine, current_inventory, machine_last_part):
    candidates = []
    for p in machine_fixed_parts.get(machine, []):
        daily = effective_daily(p)
        r_val = rate.get(p, 1)
        inv   = current_inventory.get(p, 0)
        if daily <= 0:
            continue
        candidates.append((p, inv / daily, daily / r_val if r_val > 0 else 0))
    if not candidates:
        return None
    candidates.sort(key=lambda x: (round(x[1], 4), -x[2]))
    if len(candidates) >= 2:
        top, second = candidates[0], candidates[1]
        if abs(top[1] - second[1]) < 0.01 and machine_last_part.get(machine) == top[0]:
            return second[0]
    return candidates[0][0]

def fixed_part_run_hours(part, phase):
    daily = effective_daily(part)
    r_val = rate.get(part, 1)
    if daily <= 0 or r_val <= 0:
        return AVAILABLE_HOURS
    indent_hrs = daily / r_val
    if phase == "A":
        return AVAILABLE_HOURS_EXTENDED if indent_hrs > 20.0 else AVAILABLE_HOURS
    else:
        return max(MIN_RUN_HOURS, indent_hrs)

# =============================================================
# SECTION 7B — TERMINAL CONSTRAINT
# =============================================================

def _build_part_terminals(df):
    result = {}
    if df is None or df.empty:
        return result
    part_col = next((c for c in df.columns if str(c).strip().lower() in ("part", "material")), None)
    if part_col is None:
        return result
    terminal_cols = [c for c in df.columns if str(c).strip().lower() not in ("part", "material")]
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        p = str(part).strip()
        terminals = [
            str(row[col]).strip().upper()
            for col in terminal_cols
            if pd.notna(row[col]) and str(row[col]).strip() not in ("", "nan")
        ]
        if terminals:
            result[p] = terminals
    return result

def _build_terminal_status(df):
    result = {}
    if df is None or df.empty:
        return result
    term_col = next((c for c in df.columns if str(c).strip().lower() == "terminal"), None)
    inv_col  = next((c for c in df.columns if str(c).strip().lower() == "inventory"), None)
    if term_col is None or inv_col is None:
        return result
    for _, row in df.iterrows():
        t = row[term_col]
        v = row[inv_col]
        if pd.isna(t) or str(t).strip() == "":
            continue
        key = str(t).strip().upper()
        try:
            qty = float(v) if pd.notna(v) else 0.0
        except (ValueError, TypeError):
            qty = 0.0
        result[key] = qty
    return result

part_terminals  = _build_part_terminals(hz_terminals_raw)
terminal_status = _build_terminal_status(hz_terminal_avail_raw)

if terminal_status:
    zero_count  = sum(1 for v in terminal_status.values() if v <= 0)
    avail_count = len(terminal_status) - zero_count
    print(f"  Terminals loaded : {len(terminal_status)} total  |  "
          f"{avail_count} with stock  |  {zero_count} at ZERO inventory")
else:
    print(f"  Terminals        : no data loaded — constraint inactive")


def terminal_blocked(part):
    required_terminals = part_terminals.get(part, [])
    if not required_terminals:
        return False, "", False

    dem_d   = demand_daily_raw.get(part, 0.0)
    ind_d   = indent_daily.get(part, 0.0)
    req_qty = max(dem_d, ind_d)

    r_val = rate.get(part, 0.0)
    min_run_qty = MIN_RUN_HOURS * r_val if r_val > 0 else 0.0
    effective_required = max(req_qty, min_run_qty)
    hard_floor   = effective_required * (1.0 - TERMINAL_RELAXATION_PCT)

    hard_blocking = []
    soft_blocking = []

    for t in required_terminals:
        t_inv = terminal_status.get(t, 0.0)
        if t_inv < hard_floor:
            hard_blocking.append(
                f"{t}(inv={t_inv:.0f} < hard_floor={hard_floor:.0f})"
            )
        elif t_inv < effective_required:
            soft_blocking.append(
                f"{t}(inv={t_inv:.0f} within {int(TERMINAL_RELAXATION_PCT*100)}% of "
                f"required={effective_required:.0f})"
            )

    if hard_blocking:
        return True, f"Terminal HARD BLOCK: {', '.join(hard_blocking)}", False
    if soft_blocking:
        return False, f"Terminal RELAXED ({int(TERMINAL_RELAXATION_PCT*100)}%): {', '.join(soft_blocking)}", True
    return False, "", False


def terminal_coverage_ratio(part):
    required_terminals = part_terminals.get(part, [])
    if not required_terminals:
        return 1.0
    dem_d = demand_daily_raw.get(part, 0.0)
    ind_d = indent_daily.get(part, 0.0)
    req_qty = max(dem_d, ind_d)
    r_val = rate.get(part, 0.0)
    min_run_qty = MIN_RUN_HOURS * r_val if r_val > 0 else 0.0
    effective_required = max(req_qty, min_run_qty)
    if effective_required <= 0:
        return 1.0
    ratios = [min(1.0, terminal_status.get(t, 0.0) / effective_required)
              for t in required_terminals]
    return round(min(ratios), 4)


def get_terminal_detail_for_part(part):
    required_terminals = part_terminals.get(part, [])
    if not required_terminals:
        return {
            "Terminals_Required": "None",
            "Terminal_Required_Qty": 0,
            "Terminal_Hard_Floor": 0,
            "Terminal_Status_Detail": "No terminals required",
            "Terminal_Blocked": "No",
            "Terminal_Relaxed": "No",
            "Terminal_Coverage_Ratio": 1.0,
            "Terminal_Block_Reason": "—",
        }
    dem_d   = demand_daily_raw.get(part, 0.0)
    ind_d   = indent_daily.get(part, 0.0)
    req_qty = max(dem_d, ind_d)
    r_val   = rate.get(part, 0.0)
    min_run_qty = MIN_RUN_HOURS * r_val if r_val > 0 else 0.0
    effective_required = max(req_qty, min_run_qty)
    hard_floor = effective_required * (1.0 - TERMINAL_RELAXATION_PCT)

    statuses = []
    for t in required_terminals:
        t_inv = terminal_status.get(t, None)
        if t_inv is None:
            statuses.append(f"{t}:MISSING")
        elif t_inv < hard_floor:
            statuses.append(f"{t}:inv={t_inv:.0f}:HARD_BLOCK(need>={hard_floor:.0f})")
        elif t_inv < effective_required:
            statuses.append(f"{t}:inv={t_inv:.0f}:RELAXED(need>={effective_required:.0f})")
        else:
            statuses.append(f"{t}:inv={t_inv:.0f}:OK")

    blocked, reason, relaxed = terminal_blocked(part)
    return {
        "Terminals_Required": ", ".join(required_terminals),
        "Terminal_Required_Qty": round(effective_required, 2),
        "Terminal_Hard_Floor": round(hard_floor, 2),
        "Terminal_Status_Detail": " | ".join(statuses),
        "Terminal_Blocked": "YES" if blocked else "No",
        "Terminal_Relaxed": "YES" if relaxed else "No",
        "Terminal_Coverage_Ratio": terminal_coverage_ratio(part),
        "Terminal_Block_Reason": reason if (blocked or relaxed) else "—",
    }

# =============================================================
# SECTION 7C — SKIP / ELIGIBILITY RULES
# =============================================================

def should_skip(part):
    ind_daily_v  = indent_daily.get(part, 0.0)
    dem_daily_v  = demand_daily_raw.get(part, 0.0)
    eff_daily_v  = effective_daily(part)
    monthly      = indent_monthly.get(part, 0.0)
    r            = rate.get(part, 1.0)
    inv          = inventory.get(part, 0.0)

    has_demand     = dem_daily_v > 0.0
    has_demand_gap = has_demand and inv < dem_daily_v

    if has_demand_gap:
        t_blocked, t_reason, _ = terminal_blocked(part)
        if t_blocked:
            return True, t_reason
        return False, ""

    if not has_demand:
        if ind_daily_v <= MIN_DAILY_INDENT:
            return True, f"Daily indent {ind_daily_v:.2f} <= {MIN_DAILY_INDENT} threshold (no demand)"
        indent_hrs = monthly / r if r > 0 else 0.0
        if indent_hrs <= MIN_INDENT_HOURS:
            return True, f"Monthly indent = {indent_hrs:.2f}h <= {MIN_INDENT_HOURS}h threshold (no demand)"

    if eff_daily_v > 0 and inv >= TARGET_DAYS * eff_daily_v:
        if has_demand and not is_demand_met(part, inv, 0):
            return False, ""
        return True, (
            f"Inventory ({inv:.0f}) >= {TARGET_DAYS}-day target "
            f"({TARGET_DAYS * eff_daily_v:.0f} pcs) — at ceiling"
        )

    t_blocked, t_reason, _ = terminal_blocked(part)
    if t_blocked:
        return True, t_reason

    return False, ""


def is_hard_skip(part):
    t_blocked, _, _ = terminal_blocked(part)
    return t_blocked

# =============================================================
# SECTION 7D — CHANGEOVER TIMES
# =============================================================

def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

hz_changeover          = build_changeover_dict(hz_co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0

# =============================================================
# SECTION 7E — MACHINE PART COUNT
# =============================================================

def build_machine_part_count(df):
    mpc = {}
    machine_col = next((c for c in df.columns if str(c).strip().lower() == "machine"), None)
    count_col   = next((c for c in df.columns if str(c).strip().lower() == "part_count"), None)
    if machine_col is None or count_col is None:
        print("  WARNING: HZ_Machine_Part_Count missing columns")
        return {}
    for _, row in df.iterrows():
        m = str(row[machine_col]).strip()
        v = row[count_col]
        if m and pd.notna(v):
            try:
                mpc[m] = int(float(v))
            except (ValueError, TypeError):
                pass
    return mpc

machine_part_count = build_machine_part_count(hz_machine_count_raw)
max_part_count     = max(machine_part_count.values(), default=1) or 1
print(f"  Machine part counts loaded: {len(machine_part_count)} machines")

# =============================================================
# SECTION 7F — PART CATEGORY
# =============================================================

def build_category(df):
    cat = {}
    part_col = next((c for c in df.columns if str(c).strip().lower() == "part"), None)
    cat_col  = next((c for c in df.columns if str(c).strip().lower() == "category"), None)
    if part_col is None:
        return cat
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        val = "Stranger"
        if cat_col and pd.notna(row[cat_col]):
            val = str(row[cat_col]).strip().capitalize()
            if val not in ("Runner", "Repeater", "Stranger"):
                val = "Stranger"
        cat[str(part).strip()] = val
    return cat

part_category  = build_category(hz_parts_raw)
CATEGORY_SCORE = {"Runner": 100, "Repeater": 60, "Stranger": 20}
for p in demand_daily_raw:
    if p not in part_category:
        part_category[p] = "Stranger"

# =============================================================
# SECTION 7G — SINGLE-MACHINE-PER-PART GUARD  (HZ KEY RULE)
# =============================================================

def part_is_already_assigned(part, plan):
    """HZ rule: returns True if part has ANY row in the plan already."""
    return any(r["Part"] == part for r in plan)

def parts_on_machine(machine, plan):
    return len({r["Part"] for r in plan if r["Machine"] == machine})

def machine_has_capacity_for_new_part(machine, part, plan):
    parts_on_this = {r["Part"] for r in plan if r["Machine"] == machine}
    if part in parts_on_this:
        return True
    parts_on_any = {r["Part"] for r in plan}
    if part in parts_on_any:
        return False
    return len(parts_on_this) < _dynamic_max_parts

# =============================================================
# SECTION 8 — MACHINE STATE
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        try:
            with open(MACHINE_STATE_FILE) as f:
                content = f.read().strip()
            if not content:
                os.remove(MACHINE_STATE_FILE)
                return {}
            state = json.loads(content)
            if not isinstance(state, dict):
                os.remove(MACHINE_STATE_FILE)
                return {}
            unknown = [p for p in state.values() if p not in part_color]
            if unknown:
                for p in unknown:
                    ALL_KNOWN_COLORS[p] = "NEEDS_PURGE"
            print(f"  Machine state loaded  ({len(state)} machines)")
            return state
        except Exception as e:
            print(f"  Machine state : error ({e}) — first run")
            os.remove(MACHINE_STATE_FILE)
            return {}
    print(f"  Machine state : FIRST RUN — no changeover today")
    return {}

def save_machine_state(state):
    combined = {m: p for m, p in state.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved -> '{MACHINE_STATE_FILE}'")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

hz_compat, hz_machines = build_compatibility(hz_matrix)

machine_compatible_parts = {m: [] for m in hz_machines}
for p, machines in hz_compat.items():
    for m in machines:
        if m in machine_compatible_parts:
            machine_compatible_parts[m].append(p)

# =============================================================
# SECTION 10 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv   = inventory.get(p, 0)
        daily = effective_daily(p)
        if daily == 0:
            continue
        coverage.append(inv / daily)
    if not coverage:
        return 3, "SCENARIO 3 — No active parts today"
    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < SAFETY_DAYS)
    if critical == n:
        return 0, f"SCENARIO 0 — ALL {n} active parts critical (inv < 1 day)"
    elif critical > 0:
        return 1, f"SCENARIO 1 — {critical}/{n} parts critical  |  {n-critical} have buffer"
    elif low > 0:
        return 2, f"SCENARIO 2 — {low}/{n} parts below {SAFETY_DAYS}-day safety floor"
    else:
        return 3, f"SCENARIO 3 — All {n} parts healthy (>={SAFETY_DAYS} days)"

# =============================================================
# SECTION 11 — OPD CAP
# =============================================================

def opd_cap(scenario_id):
    return min(
        {0: OPD_SCENARIO_0, 1: OPD_SCENARIO_1, 2: OPD_SCENARIO_2, 3: OPD_SCENARIO_3}.get(scenario_id, OPD_SCENARIO_2),
        TARGET_DAYS
    )

def effective_opd_cap_qty(part, scenario_id, current_inv):
    eff_d = effective_daily(part)
    dem_d = demand_daily_raw.get(part, 0.0)
    cap_q = opd_cap(scenario_id) * eff_d
    demand_floor_qty = max(0.0, dem_d - current_inv) if dem_d > 0 else 0.0
    return max(cap_q, demand_floor_qty)

# =============================================================
# SECTION 12 — PRIORITY SCORING
# =============================================================

def compute_priority_scores(active_parts):
    rows = []
    for p in active_parts:
        inv      = inventory.get(p, 0)
        daily    = effective_daily(p)
        cat      = part_category.get(p, "Stranger")
        days_cov = inv / daily if daily > 0 else 999.0
        gap_score    = min(1.0, max(0.0, (TARGET_DAYS - days_cov) / TARGET_DAYS))
        velocity_raw = daily / max(float(inv), 1.0) if daily > 0 else 0.0
        has_demand_gap = demand_daily_raw.get(p, 0) > 0 and inv < demand_daily_raw.get(p, 0)
        rows.append({
            "part": p, "inv": inv, "daily": daily, "days_cov": days_cov, "cat": cat,
            "gap_score": gap_score, "velocity_raw": velocity_raw,
            "has_demand_gap": has_demand_gap,
        })
    if not rows:
        return {}, []
    max_daily    = max(r["daily"]        for r in rows) or 1.0
    max_velocity = max(r["velocity_raw"] for r in rows) or 1.0
    scores, score_rows = {}, []
    for r in rows:
        p = r["part"]
        gap_pct      = r["gap_score"] * 100.0
        velocity_pct = (r["velocity_raw"] / max_velocity) * 100.0
        urgency_score  = 0.60 * gap_pct + 0.40 * velocity_pct
        category_score = CATEGORY_SCORE.get(r["cat"], 20)
        indent_score   = (r["daily"] / max_daily) * 100.0
        final_score = (
            W_URGENCY  * urgency_score +
            W_CATEGORY * category_score +
            W_INDENT   * indent_score
        )
        if r["has_demand_gap"]:
            final_score += 200.0
        scores[p] = round(final_score, 2)
        _, t_reason, t_relaxed = terminal_blocked(p)
        inv_today = inventory.get(p, 0)
        dem_d     = demand_daily_raw.get(p, 0.0)
        score_rows.append({
            "Part": p, "Category": r["cat"],
            "Color": part_color.get(p, "UNKNOWN"),
            "Fixed_Machine": part_fixed_machine.get(p, "—"),
            "Inventory_Now": round(r["inv"], 0),
            "Demand_Daily": round(dem_d, 2),
            "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
            "Effective_Daily": round(r["daily"], 2),
            "Demand_Driver": demand_driver(p),
            "Demand_Gap_Today": round(max(0.0, dem_d - inv_today), 2),
            "Terminal_Coverage_Ratio": terminal_coverage_ratio(p),
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Days_Coverage": round(r["days_cov"], 2),
            "Buffer_Status": (
                "CRITICAL"     if r["days_cov"] < 1 else
                "BELOW_SAFETY" if r["days_cov"] < SAFETY_DAYS else
                "BUILDING"     if r["days_cov"] < TARGET_DAYS else
                "AT_TARGET"
            ),
            "Final_Score": round(final_score, 2),
            "Demand_Boost_Applied": "YES" if r["has_demand_gap"] else "No",
            "HZ_Single_Machine_Rule": "One machine per part — enforced",
        })
    return scores, score_rows

# =============================================================
# SECTION 13 — COLOUR-AWARE CHANGEOVER HELPER
# =============================================================

def _co_hrs_for(part, machine, machine_last_part):
    last = machine_last_part.get(machine)
    if last is None or last == part:
        return 0.0
    base_co    = hz_changeover.get(machine, DEFAULT_CHANGEOVER_HRS)
    last_color = ALL_KNOWN_COLORS.get(last, "UNKNOWN")
    new_color  = part_color.get(part, "UNKNOWN")
    if last_color == "NEEDS_PURGE":
        return base_co + COLOR_PURGE_HRS
    purge = (
        COLOR_PURGE_HRS
        if last_color != new_color and last_color not in ("UNKNOWN",) and new_color not in ("UNKNOWN",)
        else 0.0
    )
    return base_co + purge

# =============================================================
# SECTION 14 — MACHINE RANKER  (BUG FIX: returns tuple)
# =============================================================

def rank_machines(part, machines_to_try, machine_hours,
                  machine_last_part, inv_days, plan,
                  exclude_fixed_machines=True):
    """
    Returns (ranked_list, runner_lock_bool).
    Enforces single-machine-per-part.
    """
    assigned_machines = {r["Machine"] for r in plan if r["Part"] == part}
    if assigned_machines:
        existing_m = list(assigned_machines)[0]
        if existing_m in machines_to_try:
            used = machine_hours.get(existing_m, 0)
            free = round(AVAILABLE_HOURS - used, 4)
            if free >= 0.05:
                return [(existing_m, 0.0, free, -1.0)], False
        return [], False

    category  = part_category.get(part, "Stranger")
    new_color = part_color.get(part, "UNKNOWN")
    fixed_m   = part_fixed_machine.get(part)
    is_fixed  = fixed_m is not None
    runner_lock = (category == "Runner" and inv_days < RUNNER_PRIORITY_DAYS and not is_fixed)

    effective_candidates = []
    for m in machines_to_try:
        if exclude_fixed_machines and m in machine_fixed_parts:
            if not is_fixed:
                continue
            elif m != fixed_m:
                continue
        effective_candidates.append(m)

    if is_fixed and fixed_m in effective_candidates:
        used_f = machine_hours.get(fixed_m, 0)
        free_f = round(AVAILABLE_HOURS - used_f, 4)
        co_f   = _co_hrs_for(part, fixed_m, machine_last_part)
        eff_f  = round(free_f - co_f, 4)
        if eff_f >= MIN_RUN_HOURS and machine_has_capacity_for_new_part(fixed_m, part, plan):
            fallback, _ = _rank_normal(
                part, [m for m in effective_candidates if m != fixed_m],
                machine_hours, machine_last_part, new_color, runner_lock, plan
            )
            return [(fixed_m, co_f, eff_f, -1.0)] + fallback, runner_lock
        remaining = [m for m in effective_candidates if m != fixed_m]
        ranked, rl = _rank_normal(part, remaining, machine_hours,
                                   machine_last_part, new_color, runner_lock, plan)
        return ranked, rl

    return _rank_normal(part, effective_candidates, machine_hours,
                        machine_last_part, new_color, runner_lock, plan)


def _rank_normal(part, machines_to_try, machine_hours,
                 machine_last_part, new_color, runner_lock, plan):
    """Returns (ranked_list, runner_lock_bool)."""
    ranked = []
    for m in machines_to_try:
        if not machine_has_capacity_for_new_part(m, part, plan):
            continue
        used = machine_hours.get(m, 0)
        free = round(AVAILABLE_HOURS - used, 4)
        if free < MIN_RUN_HOURS:
            continue
        last = machine_last_part.get(m)
        if runner_lock and last != part:
            continue
        if last is None or last == part:
            co_hrs = 0.0; color_bonus = 0.0
        else:
            base_co    = hz_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
            last_color = ALL_KNOWN_COLORS.get(last, "UNKNOWN")
            same_color = (last_color == new_color
                          and last_color not in ("UNKNOWN", "NEEDS_PURGE")
                          and new_color not in ("UNKNOWN",))
            purge      = 0.0 if same_color else (
                COLOR_PURGE_HRS
                if last_color not in ("UNKNOWN",) and new_color not in ("UNKNOWN",)
                else 0.0
            )
            co_hrs     = base_co + purge
            color_bonus = -0.08 if same_color else 0.0
        effective_free = round(free - co_hrs, 4)
        if effective_free < MIN_RUN_HOURS:
            continue
        part_count      = machine_part_count.get(m, max_part_count)
        count_score     = part_count / max_part_count
        co_penalty      = (co_hrs / AVAILABLE_HOURS) * 0.3
        util_penalty    = (used  / AVAILABLE_HOURS) * 0.2
        same_part_bonus = -0.15 if (last == part) else 0.0
        cost            = count_score + co_penalty + util_penalty + same_part_bonus + color_bonus
        ranked.append((m, co_hrs, effective_free, cost))
    ranked.sort(key=lambda x: x[3])
    return ranked, runner_lock

_phase_a_machines: set = set()

# =============================================================
# SECTION 14A — PLAN HELPERS
# =============================================================

def _get_part_total_qty(part, plan):
    return sum(float(r.get("Production_Qty", 0)) for r in plan if r["Part"] == part)

def _is_indent_met(part, plan):
    daily = effective_daily(part)
    if daily <= 0:
        return True
    total_qty = _get_part_total_qty(part, plan)
    inv_now   = inventory.get(part, 0)
    return (inv_now + total_qty) >= (daily - 0.5)

def _current_co_count(plan):
    return sum(1 for r in plan if r.get("Changeover") == "Yes")

def _make_plan_row(part, machine, run_hrs, co_hrs, qty, scenario_id,
                   type_label, role_label, runner_lock=False, phase=1):
    daily    = effective_daily(part)
    monthly  = effective_monthly(part)
    r_val    = rate.get(part, 1)
    inv_now  = inventory.get(part, 0)
    color    = part_color.get(part, "UNKNOWN")
    fixed_m  = part_fixed_machine.get(part)
    last     = machine_state.get(machine)
    l_color  = ALL_KNOWN_COLORS.get(last, "UNKNOWN") if last else "UNKNOWN"
    has_purge = (last is not None and last != part and color != l_color
                 and color != "UNKNOWN" and l_color not in ("UNKNOWN", "NEEDS_PURGE"))
    fixed_used = "YES" if (fixed_m and machine == fixed_m) else ("FALLBACK" if fixed_m else "N/A")
    dem_d = demand_daily_raw.get(part, 0.0)
    indent_met_flag = (
        "YES" if (inv_now + qty) >= (daily - 0.5)
        else f"NO — need {daily:.2f}/d, have {inv_now+qty:.0f} pcs"
    )
    demand_met_flag = "YES" if is_demand_met(part, inv_now, qty) else f"NO — need {dem_d:.2f}/d"
    _, t_reason, t_relaxed = terminal_blocked(part)
    return {
        "Part": part,
        "Color": color,
        "Category": part_category.get(part, "Stranger"),
        "Fixed_Machine": fixed_m or "—",
        "Fixed_Used": fixed_used,
        "Machine": machine,
        "Run_Hours": round(run_hrs, 3),
        "Changeover_Hrs": round(co_hrs, 3),
        "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour": round(r_val, 2),
        "Production_Qty": qty,
        "Inventory_Before": round(inv_now, 0),
        "Demand_Today_Required": round(max(0.0, dem_d - inv_now), 2),
        "Demand_Daily": round(dem_d, 2),
        "Indent_Daily": round(indent_daily.get(part, 0.0), 2),
        "Effective_Daily": round(daily, 2),
        "Demand_Driver": demand_driver(part),
        "Monthly_Indent": round(monthly, 0),
        "Today_Target": round(today_target_qty.get(part, 0), 0),
        "Changeover": "No" if co_hrs == 0 else "Yes",
        "Color_Purge": "Yes" if has_purge else "No",
        "Terminal_Relaxed": "YES" if t_relaxed else "No",
        "Terminal_Note": t_reason if t_relaxed else "—",
        "Terminal_Reminder": "—",
        "Type": type_label,
        "Role": role_label,
        "HZ_Single_Machine": "ENFORCED",
        "Runner_Lock": "YES" if runner_lock else "No",
        "Priority_Score": 0,
        "Phase": phase,
        "Indent_Met": indent_met_flag,
        "Demand_Met": demand_met_flag,
        "Stagger_Adjusted": "No",
    }

# =============================================================
# SECTION 14B — INTRA-MACHINE CO RESEQUENCING
# =============================================================

def resequence_machine_rows(plan, machine_last_part_yesterday):
    machine_rows = defaultdict(list)
    other_rows   = []
    for row in plan:
        m = row.get("Machine")
        if m in hz_machines:
            machine_rows[m].append(row)
        else:
            other_rows.append(row)
    resequenced_plan = []
    for m in hz_machines:
        rows = machine_rows.get(m, [])
        if len(rows) <= 1:
            resequenced_plan.extend(rows)
            continue
        yesterday_part = machine_last_part_yesterday.get(m)
        ordered   = []
        remaining = list(rows)
        seed = None
        if yesterday_part:
            for r in remaining:
                if r["Part"] == yesterday_part:
                    seed = r
                    break
        if seed is None:
            seed = max(remaining, key=lambda r: float(r.get("Priority_Score", 0) or 0))
        ordered.append(seed)
        remaining.remove(seed)
        while remaining:
            last_part      = ordered[-1]["Part"]
            last_color_val = part_color.get(last_part, "UNKNOWN")
            def _co_cost(r, _lc=last_color_val):
                p = r["Part"]
                c = part_color.get(p, "UNKNOWN")
                if p == last_part:
                    return -1.0
                base  = hz_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
                purge = (COLOR_PURGE_HRS if _lc not in ("UNKNOWN", "NEEDS_PURGE") and c not in ("UNKNOWN",) and _lc != c else 0.0)
                return base + purge
            remaining.sort(key=_co_cost)
            ordered.append(remaining.pop(0))
        for i, row in enumerate(ordered):
            p         = row["Part"]
            prev_part = yesterday_part if i == 0 else ordered[i - 1]["Part"]
            if prev_part is None or prev_part == p:
                new_co = 0.0
            else:
                base_co    = hz_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
                prev_color = ALL_KNOWN_COLORS.get(prev_part, "UNKNOWN")
                new_color  = part_color.get(p, "UNKNOWN")
                purge = (COLOR_PURGE_HRS if prev_color not in ("UNKNOWN", "NEEDS_PURGE") and new_color not in ("UNKNOWN",) and prev_color != new_color else 0.0)
                new_co = base_co + purge
            row["Changeover_Hrs"]  = round(new_co, 3)
            row["Changeover"]      = "No" if new_co == 0 else "Yes"
            row["Total_Hrs_Used"]  = round(new_co + float(row.get("Run_Hours", 0) or 0), 3)
            row["Color_Purge"] = "Yes" if new_co > hz_changeover.get(m, DEFAULT_CHANGEOVER_HRS) + 0.001 else "No"
        resequenced_plan.extend(ordered)
    resequenced_plan.extend(other_rows)
    return resequenced_plan

def reconcile_machine_hours(plan, machine_hours):
    recomputed = {m: 0.0 for m in hz_machines}
    for row in plan:
        m = row.get("Machine")
        if m in recomputed:
            recomputed[m] += float(row.get("Run_Hours", 0) or 0) + float(row.get("Changeover_Hrs", 0) or 0)
    for m in hz_machines:
        machine_hours[m] = round(recomputed[m], 4)

# =============================================================
# SECTION 15 — PASS 0: DEMAND-FIRST ASSIGNMENT  (BUG FIX: zero-rate guard)
# =============================================================

def assign_demand_for_part(part, scenario_id, machine_hours, machine_last_part,
                            current_inventory, plan, already_planned, priority_scores):
    """HZ: Satisfies today's demand gap on ONE machine only."""
    daily      = effective_daily(part)
    r_val      = rate.get(part, 0)          # FIX: default 0, guard below
    inv_now    = current_inventory.get(part, 0)
    inv_before = inventory.get(part, 0)
    dem_d      = demand_daily_raw.get(part, 0.0)
    dem_gap    = max(0.0, dem_d - inv_now)

    if dem_gap <= 0:
        return []

    if r_val <= 0:                           # FIX: guard zero/missing rate
        return []

    if part_is_already_assigned(part, plan):
        return []

    compatible = hz_compat.get(part, [])
    if not compatible:
        return []

    fixed_m = part_fixed_machine.get(part)
    _, t_note, t_relaxed = terminal_blocked(part)

    if fixed_m:
        ordered = [fixed_m] if fixed_m in compatible else []
    else:
        ordered = [m for m in compatible if m not in machine_fixed_parts]

    new_rows = []
    produced = 0.0

    for m in ordered:
        if produced >= dem_gap - 0.5:
            break
        if not machine_has_capacity_for_new_part(m, part, plan):
            continue
        if m in _phase_a_machines and m != fixed_m:
            continue
        co_hrs = _co_hrs_for(part, m, machine_last_part)
        if co_hrs > 0 and _current_co_count(plan) >= MAX_DAILY_CO and inv_now > 0:
            continue
        free   = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        eff    = round(free - co_hrs, 4)
        if eff < MIN_RUN_HOURS:
            continue

        # FIX: r_val already guarded above, safe to divide
        run_hrs = max(MIN_RUN_HOURS, min(eff, dem_gap / r_val))
        qty     = round(run_hrs * r_val, 0)

        machine_hours[m]        = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
        current_inventory[part] = round(current_inventory.get(part, 0) + qty, 0)
        machine_last_part[m]    = part
        produced               += qty

        last    = machine_state.get(m)
        l_color = ALL_KNOWN_COLORS.get(last, "UNKNOWN") if last else "UNKNOWN"
        p_color = part_color.get(part, "UNKNOWN")
        has_purge = (last is not None and last != part and p_color != l_color
                     and p_color != "UNKNOWN" and l_color not in ("UNKNOWN", "NEEDS_PURGE"))

        dem_met_flag = (
            "YES" if (inv_before + produced) >= (dem_d - 0.5)
            else f"NO — need {max(0, dem_d - (inv_before + produced)):.0f} more"
        )
        ind_met_flag = (
            "YES" if (inv_before + produced) >= (daily - 0.5)
            else f"NO — need {daily:.2f}/d"
        )

        new_rows.append({
            "Part": part,
            "Color": p_color,
            "Category": part_category.get(part, "Stranger"),
            "Fixed_Machine": fixed_m or "—",
            "Fixed_Used": "YES" if (fixed_m and m == fixed_m) else ("FALLBACK" if fixed_m else "N/A"),
            "Machine": m,
            "Run_Hours": round(run_hrs, 3),
            "Changeover_Hrs": round(co_hrs, 3),
            "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
            "Rate_Per_Hour": round(r_val, 2),
            "Production_Qty": qty,
            "Inventory_Before": round(inv_before, 0),
            "Demand_Today_Required": round(dem_gap, 2),
            "Demand_Daily": round(dem_d, 2),
            "Indent_Daily": round(indent_daily.get(part, 0.0), 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(part),
            "Monthly_Indent": round(effective_monthly(part), 0),
            "Today_Target": round(today_target_qty.get(part, 0), 0),
            "Changeover": "No" if co_hrs == 0 else "Yes",
            "Color_Purge": "Yes" if has_purge else "No",
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Terminal_Note": t_note if t_relaxed else "—",
            "Terminal_Reminder": "—",
            "Type": "DEMAND-FIRST" + (" [TERMINAL-RELAXED]" if t_relaxed else ""),
            "Role": "Primary",
            "HZ_Single_Machine": "ENFORCED",
            "Runner_Lock": "No",
            "Priority_Score": priority_scores.get(part, 0),
            "Phase": 0,
            "Indent_Met": ind_met_flag,
            "Demand_Met": dem_met_flag,
            "Stagger_Adjusted": "No",
        })
        break  # HZ RULE: one machine per part

    if new_rows:
        already_planned.add(part)
    return new_rows

# =============================================================
# SECTION 16 — PASS 2: INVENTORY BUILD  (BUG FIX: tuple unpack + plan.extend order)
# =============================================================

def assign_inventory_build(part, scenario_id, machine_hours, machine_last_part,
                            current_inventory, plan, already_planned, priority_scores):
    """HZ: Builds inventory on ONE machine only."""
    daily      = effective_daily(part)
    r_val      = rate.get(part, 1)
    inv_now    = current_inventory.get(part, 0)
    inv_before = inventory.get(part, 0)
    compatible = hz_compat.get(part, [])
    fixed_m    = part_fixed_machine.get(part)
    _, t_note, t_relaxed = terminal_blocked(part)

    if not compatible:
        return []

    # HZ RULE: if part is already on a machine, only extend that machine
    assigned_m = next((r["Machine"] for r in plan if r["Part"] == part), None)
    if assigned_m:
        _extend_part_on_machine(part, assigned_m, plan, scenario_id,
                                 machine_hours, current_inventory, r_val, daily, inv_now)
        return []

    if fixed_m:
        machines_to_rank = [fixed_m] if fixed_m in compatible else []
    else:
        machines_to_rank = [m for m in compatible if m not in machine_fixed_parts]

    if not machines_to_rank:
        return []

    inv_days        = inv_now / daily if daily > 0 else 999
    total_shortfall = max(0.0, daily - inv_now)

    new_rows = []
    produced = 0.0

    # FIX: unpack tuple — (ranked_list, runner_lock_bool)
    ranked, runner_lock = rank_machines(
        part, machines_to_rank, machine_hours, machine_last_part, inv_days, plan
    )

    for m, co, eff, _ in ranked:
        if m in _phase_a_machines and m != fixed_m:
            continue
        if co > 0 and _current_co_count(plan) >= MAX_DAILY_CO:
            continue
        run_hrs = max(MIN_RUN_HOURS, min(eff, total_shortfall / r_val if r_val > 0 else MIN_RUN_HOURS))
        qty     = round(run_hrs * r_val, 0)

        machine_hours[m]        = round(machine_hours.get(m, 0) + co + run_hrs, 4)
        current_inventory[part] = round(current_inventory.get(part, 0) + qty, 0)
        machine_last_part[m]    = part
        produced               += qty

        last    = machine_state.get(m)
        l_color = ALL_KNOWN_COLORS.get(last, "UNKNOWN") if last else "UNKNOWN"
        p_color = part_color.get(part, "UNKNOWN")
        has_purge = (last is not None and last != part and p_color != l_color
                     and p_color != "UNKNOWN" and l_color not in ("UNKNOWN", "NEEDS_PURGE"))
        fixed_used_str = "YES" if (fixed_m and m == fixed_m) else ("FALLBACK" if fixed_m else "N/A")
        dem_d = demand_daily_raw.get(part, 0.0)

        new_rows.append({
            "Part": part,
            "Color": p_color,
            "Category": part_category.get(part, "Stranger"),
            "Fixed_Machine": fixed_m or "—",
            "Fixed_Used": fixed_used_str,
            "Machine": m,
            "Run_Hours": round(run_hrs, 3),
            "Changeover_Hrs": round(co, 3),
            "Total_Hrs_Used": round(co + run_hrs, 3),
            "Rate_Per_Hour": round(r_val, 2),
            "Production_Qty": qty,
            "Inventory_Before": round(inv_before, 0),
            "Demand_Today_Required": round(max(0.0, dem_d - inv_before), 2),
            "Demand_Daily": round(dem_d, 2),
            "Indent_Daily": round(indent_daily.get(part, 0.0), 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(part),
            "Monthly_Indent": round(effective_monthly(part), 0),
            "Today_Target": round(today_target_qty.get(part, 0), 0),
            "Changeover": "No" if co == 0 else "Yes",
            "Color_Purge": "Yes" if has_purge else "No",
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Terminal_Note": t_note if t_relaxed else "—",
            "Terminal_Reminder": "—",
            "Type": "Indent-Build" + (" [TERMINAL-RELAXED]" if t_relaxed else ""),
            "Role": "Primary",
            "HZ_Single_Machine": "ENFORCED",
            "Runner_Lock": "YES" if runner_lock else "No",
            "Priority_Score": priority_scores.get(part, 0),
            "Phase": 1,
            "Indent_Met": "YES" if (inv_before + produced) >= (daily - 0.5) else f"NO — need {daily:.2f}/d",
            "Demand_Met": "YES" if is_demand_met(part, inv_before, produced) else f"NO — need {dem_d:.2f}/d",
            "Stagger_Adjusted": "No",
        })
        print(f"      INDENT-BUILD {part:26s} -> {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}")
        break  # HZ RULE: one machine per part

    if new_rows:
        already_planned.add(part)
        plan.extend(new_rows)           # FIX: add to plan FIRST ...
        assigned_m = new_rows[0]["Machine"]
        _extend_part_on_machine(        # ... THEN extend (so it can find the row)
            part, assigned_m, plan, scenario_id,
            machine_hours, current_inventory,
            r_val, daily, current_inventory.get(part, 0)
        )
    return new_rows


def _extend_part_on_machine(part, machine, plan, scenario_id, machine_hours,
                              current_inventory, r_val, daily, inv_after):
    """Extend the single plan row for this part to reach OPD cap."""
    if machine in _phase_a_machines:
        return
    cap_qty  = effective_opd_cap_qty(part, scenario_id, inv_after)
    headroom = max(0.0, cap_qty - inv_after)
    if headroom <= 0:
        return
    target_row = next((r for r in plan if r["Part"] == part and r["Machine"] == machine), None)
    if target_row is None:
        return
    free_m = round(AVAILABLE_HOURS - machine_hours.get(machine, 0), 4)
    if free_m < 0.05:
        return
    extend_hrs = min(free_m, headroom / r_val if r_val > 0 else 0)
    if extend_hrs < 0.05:
        return
    extra_qty = round(extend_hrs * r_val, 0)
    target_row["Run_Hours"]      = round(float(target_row["Run_Hours"]) + extend_hrs, 3)
    target_row["Total_Hrs_Used"] = round(float(target_row["Changeover_Hrs"]) + float(target_row["Run_Hours"]), 3)
    target_row["Production_Qty"] = round(float(target_row["Production_Qty"]) + extra_qty, 0)
    target_row["Type"] = str(target_row["Type"]) + "+OPD-Build"
    machine_hours[machine]        = round(machine_hours.get(machine, 0) + extend_hrs, 4)
    current_inventory[part]       = round(current_inventory.get(part, 0) + extra_qty, 0)
    print(f"      OPD-BUILD  {part:25s} on {machine:15s}  +{extend_hrs:.2f}h  qty+={extra_qty:.0f}")

# =============================================================
# SECTION 17 — FIXED MACHINE SCHEDULING PASS
# =============================================================

def schedule_fixed_machines(machine_hours, machine_last_part,
                              current_inventory, plan, already_planned,
                              priority_scores, scenario_id):
    print("\n" + "─"*65)
    print(f"  FIXED MACHINE SCHEDULING PASS (HZ)")
    print("─"*65)

    phase_a_machines = set()
    fixed_plan_rows  = []

    for machine, fixed_parts_list in sorted(machine_fixed_parts.items()):
        phase = fixed_machine_phase(machine, current_inventory)

        if phase == "A":
            phase_a_machines.add(machine)
            chosen = pick_fixed_part_for_today(machine, current_inventory, machine_last_part)
            if chosen is None:
                continue
            t_blk_a, t_note, t_relaxed = terminal_blocked(chosen)
            terminal_reminder_a = None
            if t_blk_a:
                dem_d_rem = demand_daily_raw.get(chosen, 0.0)
                ind_d_rem = indent_daily.get(chosen, 0.0)
                r_rem = rate.get(chosen, 0.0)
                req_rem = max(dem_d_rem, ind_d_rem, MIN_RUN_HOURS * r_rem)
                terminal_reminder_a = (
                    f"REMINDER: Terminal stock insufficient for {chosen} — "
                    f"need ~{req_rem:.0f} pcs. {t_note[:80]}"
                )
                t_blk_a = False
            daily   = effective_daily(chosen)
            r_val   = rate.get(chosen, 1)
            score   = priority_scores.get(chosen, 0)
            inv_before_chosen = inventory.get(chosen, 0)
            run_hrs_cap = fixed_part_run_hours(chosen, "A")
            qty = round(run_hrs_cap * r_val, 0)
            machine_hours[machine]      = round(run_hrs_cap, 4)
            current_inventory[chosen]   = round(current_inventory.get(chosen, 0) + qty, 0)
            machine_last_part[machine]  = chosen
            already_planned.add(chosen)
            row = _make_plan_row(chosen, machine, run_hrs_cap, 0.0, qty, scenario_id,
                                 f"Fixed-PhaseA [{run_hrs_cap}h cap]", "Primary", phase=1)
            row["Priority_Score"]   = score
            row["Inventory_Before"] = round(inv_before_chosen, 0)
            if terminal_reminder_a:
                row["Terminal_Reminder"] = terminal_reminder_a
                row["Terminal_Relaxed"]  = "YES (Fixed Override)"
            plan.append(row)
            fixed_plan_rows.append(row)
            print(f"    Phase A -> {chosen}  {run_hrs_cap:.2f}h  qty={qty:.0f}")

        else:
            for chosen in fixed_parts_list:
                if machine_hours.get(machine, 0) >= AVAILABLE_HOURS - 0.05:
                    already_planned.add(chosen)
                    continue
                t_blk_b, t_note_b, _ = terminal_blocked(chosen)
                terminal_reminder_b = None
                if t_blk_b:
                    dem_d_rem_b = demand_daily_raw.get(chosen, 0.0)
                    ind_d_rem_b = indent_daily.get(chosen, 0.0)
                    r_rem_b = rate.get(chosen, 0.0)
                    req_rem_b = max(dem_d_rem_b, ind_d_rem_b, MIN_RUN_HOURS * r_rem_b)
                    terminal_reminder_b = (
                        f"REMINDER: Terminal stock insufficient for {chosen} — "
                        f"need ~{req_rem_b:.0f} pcs."
                    )
                    t_blk_b = False
                daily   = effective_daily(chosen)
                r_val   = rate.get(chosen, 1)
                score   = priority_scores.get(chosen, 0)
                inv_before_chosen = inventory.get(chosen, 0)
                min_run_for_indent = fixed_part_run_hours(chosen, "B")
                available_now = round(AVAILABLE_HOURS - machine_hours.get(machine, 0), 4)
                effective_run = min(min_run_for_indent, available_now)
                if effective_run < MIN_RUN_HOURS:
                    already_planned.add(chosen)
                    continue
                inv_now_chosen = current_inventory.get(chosen, 0)
                cap_qty  = effective_opd_cap_qty(chosen, scenario_id, inv_now_chosen)
                headroom = max(0.0, cap_qty - inv_now_chosen)
                if headroom > 0 and r_val > 0:
                    effective_run = min(available_now, max(effective_run, headroom / r_val))
                    effective_run = max(effective_run, MIN_RUN_HOURS)
                qty = round(effective_run * r_val, 0)
                machine_hours[machine]    = round(machine_hours.get(machine, 0) + effective_run, 4)
                current_inventory[chosen] = round(current_inventory.get(chosen, 0) + qty, 0)
                machine_last_part[machine] = chosen
                already_planned.add(chosen)
                _, t_note, t_relaxed = terminal_blocked(chosen)
                row = _make_plan_row(chosen, machine, effective_run, 0.0, qty, scenario_id,
                                     "Fixed-PhaseB", "Primary", phase=1)
                row["Priority_Score"]   = score
                row["Inventory_Before"] = round(inv_before_chosen, 0)
                if terminal_reminder_b:
                    row["Terminal_Reminder"] = terminal_reminder_b
                    row["Terminal_Relaxed"]  = "YES (Fixed Override)"
                plan.append(row)
                fixed_plan_rows.append(row)
                print(f"    Phase B -> {chosen}  {effective_run:.2f}h  qty={qty:.0f}")

    print(f"\n  Fixed machine pass: {len(phase_a_machines)} Phase A  |  "
          f"{len(machine_fixed_parts) - len(phase_a_machines)} Phase B")
    return phase_a_machines, fixed_plan_rows

# =============================================================
# SECTION 17B — FAIR CAPACITY PRE-DISTRIBUTION
# =============================================================

def pre_distribute_parts_to_machines(eligible_parts, machines_list):
    global _dynamic_max_parts

    eligible_set   = set(eligible_parts)
    fixed_machines = set(machine_fixed_parts.keys())

    machine_eligible = {}
    for m in machines_list:
        if m in fixed_machines:
            continue
        compat = [p for p in machine_compatible_parts.get(m, []) if p in eligible_set]
        if compat:
            machine_eligible[m] = set(compat)

    active_machines = list(machine_eligible.keys())
    n_machines = len(active_machines)
    n_parts    = len(eligible_parts)

    if n_machines == 0:
        return {m: [] for m in machines_list}, MAX_PARTS_PER_MACHINE

    dynamic_max = math.ceil(n_parts / n_machines) if n_machines > 0 else MAX_PARTS_PER_MACHINE
    dynamic_max = max(1, min(dynamic_max, MAX_PARTS_PER_MACHINE))
    _dynamic_max_parts = dynamic_max

    print(f"\n  HZ PRE-DISTRIBUTION: {n_parts} eligible parts / {n_machines} active machines "
          f"=> {dynamic_max} parts/machine cap")

    def _urgency_key(p):
        inv_p  = inventory.get(p, 0)
        dem_d  = demand_daily_raw.get(p, 0.0)
        daily  = effective_daily(p)
        dem_gap = max(0.0, dem_d - inv_p)
        days_cov = inv_p / daily if daily > 0 else 999
        has_gap  = 1 if dem_gap > 0 else 0
        return (-has_gap, -dem_gap, days_cov, -terminal_coverage_ratio(p))

    sorted_parts = sorted(eligible_parts, key=_urgency_key)
    machine_assignment = {m: [] for m in machines_list}

    for p in sorted_parts:
        compatible = [
            m for m in active_machines
            if p in machine_eligible.get(m, set())
            and len(machine_assignment[m]) < dynamic_max
        ]
        if not compatible:
            continue
        best_m = min(compatible, key=lambda m: len(machine_assignment[m]))
        machine_assignment[best_m].append(p)

    for m in active_machines:
        assigned = machine_assignment[m]
        print(f"    {m:<20} assigned {len(assigned)} parts: "
              f"{', '.join(assigned[:5])}{'...' if len(assigned) > 5 else ''}")

    return machine_assignment, dynamic_max

# =============================================================
# SECTION 18 — RUNNER PRIORITY ENFORCEMENT
# =============================================================

def enforce_runner_priority(plan, machine_hours, machine_last_part,
                             current_inventory, already_planned,
                             priority_scores, inventory_start_of_day):
    print("\n" + "─"*65)
    print(f"  RUNNER PRIORITY ENFORCEMENT (HZ — single machine)")
    print("─"*65)
    runner_priority_log = []

    critical_runners = []
    for part in hz_compat:
        if part in already_planned:
            continue
        if part_category.get(part, "Stranger") != "Runner":
            continue
        daily = effective_daily(part)
        r_val = rate.get(part, 1)
        if daily <= 0 or r_val <= 0:
            continue
        inv_now = current_inventory.get(part, 0)
        if inv_now / daily >= RUNNER_PRIORITY_DAYS:
            continue
        skip, _ = should_skip(part)
        if skip:
            continue
        critical_runners.append(part)

    if not critical_runners:
        print(f"  No critical unplanned Runners found")
        return runner_priority_log

    critical_runners.sort(key=lambda p: effective_daily(p), reverse=True)
    print(f"  Critical Runners: {len(critical_runners)}")

    for runner in critical_runners:
        r_daily   = effective_daily(runner)
        r_inv     = current_inventory.get(runner, 0)
        r_inv_sod = inventory_start_of_day.get(runner, r_inv)
        r_inv_b   = inventory.get(runner, 0)
        r_rate    = rate.get(runner, 1)
        r_score   = priority_scores.get(runner, 0)
        r_monthly = effective_monthly(runner)
        fixed_m   = part_fixed_machine.get(runner)
        r_days_now = r_inv / r_daily if r_daily > 0 else 0
        r_days_sod = r_inv_sod / r_daily if r_daily > 0 else 0
        _, t_note, t_relaxed = terminal_blocked(runner)
        dem_d = demand_daily_raw.get(runner, 0.0)

        shortfall_qty = max(0.0, r_daily - r_inv)
        hours_needed  = max(MIN_RUN_HOURS, shortfall_qty / r_rate if r_rate > 0 else MIN_RUN_HOURS)

        if part_is_already_assigned(runner, plan):
            continue

        compatible_machines = list(hz_compat.get(runner, []))
        if fixed_m and fixed_m in compatible_machines:
            ordered_machines = [fixed_m] + [m for m in compatible_machines if m != fixed_m]
        else:
            ordered_machines = [m for m in compatible_machines if m not in machine_fixed_parts]

        placed = False
        for m in ordered_machines:
            if not machine_has_capacity_for_new_part(m, runner, plan):
                continue
            co_hrs   = _co_hrs_for(runner, m, machine_last_part)
            used_hrs = machine_hours.get(m, 0)
            free_hrs = round(AVAILABLE_HOURS - used_hrs, 4)
            eff_free = round(free_hrs - co_hrs, 4)

            if eff_free >= hours_needed:
                run_h = max(MIN_RUN_HOURS, min(eff_free, hours_needed))
                qty   = round(run_h * r_rate, 0)
                fixed_used = (fixed_m is not None and m == fixed_m)
                machine_hours[m]          = round(used_hrs + co_hrs + run_h, 4)
                current_inventory[runner] = round(current_inventory.get(runner, 0) + qty, 0)
                machine_last_part[m]      = runner
                already_planned.add(runner)
                purge = co_hrs > hz_changeover.get(m, DEFAULT_CHANGEOVER_HRS) + 0.001
                dem_met = is_demand_met(runner, r_inv_b, qty)
                plan.append({
                    "Part": runner, "Color": part_color.get(runner, "UNKNOWN"),
                    "Category": "Runner",
                    "Fixed_Machine": fixed_m or "—",
                    "Fixed_Used": "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
                    "Machine": m, "Run_Hours": round(run_h, 3),
                    "Changeover_Hrs": round(co_hrs, 3),
                    "Total_Hrs_Used": round(co_hrs + run_h, 3),
                    "Rate_Per_Hour": round(r_rate, 2),
                    "Production_Qty": qty,
                    "Inventory_Before": round(r_inv_b, 0),
                    "Demand_Today_Required": round(max(0.0, dem_d - r_inv_b), 2),
                    "Demand_Daily": round(dem_d, 2),
                    "Indent_Daily": round(indent_daily.get(runner, 0.0), 2),
                    "Effective_Daily": round(r_daily, 2),
                    "Demand_Driver": demand_driver(runner),
                    "Monthly_Indent": round(r_monthly, 0),
                    "Today_Target": round(today_target_qty.get(runner, 0), 0),
                    "Changeover": "No" if co_hrs == 0 else "Yes",
                    "Color_Purge": "Yes" if purge else "No",
                    "Terminal_Relaxed": "YES" if t_relaxed else "No",
                    "Terminal_Note": t_note if t_relaxed else "—",
                    "Terminal_Reminder": "—",
                    "Type": "Runner-Priority" + (" [TERMINAL-RELAXED]" if t_relaxed else ""),
                    "Role": "Primary",
                    "HZ_Single_Machine": "ENFORCED",
                    "Runner_Lock": "No",
                    "Priority_Score": r_score, "Phase": 1,
                    "Indent_Met": "YES" if qty >= shortfall_qty else f"NO — need {r_daily:.2f}/d",
                    "Demand_Met": "YES" if dem_met else "NO",
                    "Stagger_Adjusted": "No",
                })
                runner_priority_log.append({
                    "Runner_Part": runner, "Demand_Driver": demand_driver(runner),
                    "Runner_Days_SOD": round(r_days_sod, 2),
                    "Runner_Days_Now": round(r_days_now, 2),
                    "Fixed_Machine": fixed_m or "—",
                    "Runner_Daily": round(r_daily, 2),
                    "Runner_Shortfall": round(shortfall_qty, 0),
                    "Machine_Assigned": m,
                    "Hours_Assigned": round(run_h, 3), "Qty_Produced": qty,
                    "Terminal_Relaxed": "YES" if t_relaxed else "No",
                    "Result": "PLANNED",
                })
                print(f"    {runner:30s} -> {m}  run={run_h:.2f}h  qty={qty:.0f}")
                placed = True
                break

        if not placed:
            runner_priority_log.append({
                "Runner_Part": runner, "Demand_Driver": demand_driver(runner),
                "Runner_Days_SOD": round(r_days_sod, 2),
                "Runner_Days_Now": round(r_days_now, 2),
                "Fixed_Machine": fixed_m or "—",
                "Runner_Daily": round(r_daily, 2),
                "Runner_Shortfall": round(shortfall_qty, 0),
                "Machine_Assigned": "—",
                "Hours_Assigned": 0, "Qty_Produced": 0,
                "Terminal_Relaxed": "YES" if t_relaxed else "No",
                "Result": "FAILED — no eligible machine",
            })

    return runner_priority_log

# =============================================================
# SECTION 19 — UTILIZATION ENFORCER  (BUG FIX: single-machine guard)
# =============================================================

def _extend_existing_on_machine(m, plan, machine_hours, current_inventory,
                                 priority_scores, ceiling_days, remaining):
    consumed = 0.0
    parts_on_m = sorted(
        [row for row in plan if row["Machine"] == m],
        key=lambda r: float(priority_scores.get(r["Part"], 0) or 0),
        reverse=True,
    )
    for row in parts_on_m:
        if remaining < 0.001:
            break
        p_ext   = row["Part"]
        if is_hard_skip(p_ext):
            continue
        r_ext   = rate.get(p_ext, 1)
        daily_p = effective_daily(p_ext)
        inv_now = current_inventory.get(p_ext, 0)
        headroom = max(0.0, ceiling_days * daily_p - inv_now) if daily_p > 0 else 0
        ext_hrs  = min(remaining, headroom / r_ext if r_ext > 0 else 0)
        if ext_hrs < 0.001:
            continue
        extra_qty = round(ext_hrs * r_ext, 0)
        row["Run_Hours"]      = round(float(row.get("Run_Hours", 0)) + ext_hrs, 3)
        row["Production_Qty"] = round(float(row.get("Production_Qty", 0)) + extra_qty, 0)
        row["Total_Hrs_Used"] = round(float(row.get("Changeover_Hrs", 0)) + float(row["Run_Hours"]), 3)
        row["Type"] = str(row.get("Type", "")) + f"+Ext{ceiling_days}d"
        machine_hours[m]         = round(machine_hours.get(m, 0) + ext_hrs, 4)
        current_inventory[p_ext] = round(current_inventory.get(p_ext, 0) + extra_qty, 0)
        remaining = round(remaining - ext_hrs, 4)
        consumed += ext_hrs
    return consumed, remaining


def utilization_enforcer(plan, machine_hours, machine_last_part,
                          all_parts, already_planned,
                          current_inventory, scenario_id, priority_scores):
    print(f"\n  UTILIZATION ENFORCER (HZ — single machine per part, {MAX_DAILY_CO} CO cap)")
    micro_idle_log = []
    floor_hrs = AVAILABLE_HOURS * (UTIL_TARGET_PCT / 100.0)
    machines_by_util = sorted(hz_machines, key=lambda m: machine_hours.get(m, 0))

    for m in machines_by_util:
        if m in _phase_a_machines:
            continue

        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining < 0.05:
            continue
        last_on_m  = machine_last_part.get(m)
        last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"

        # Extend existing parts to fill up to floor
        if machine_hours.get(m, 0) < floor_hrs:
            needed = round(floor_hrs - machine_hours.get(m, 0), 4)
            _, remaining = _extend_existing_on_machine(
                m, plan, machine_hours, current_inventory,
                priority_scores, opd_cap(scenario_id), min(needed, remaining))
            remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)

        if remaining < 0.05:
            continue

        # Try filling with unplanned parts
        if remaining >= MIN_RUN_HOURS:
            machine_compat = machine_compatible_parts.get(m, [])
            unplanned = []
            for p in machine_compat:
                if rate.get(p, 0) <= 0:
                    continue
                if indent_monthly.get(p, 0) <= 0 and demand_daily_raw.get(p, 0) <= 0:
                    continue
                t_blk, _, _ = terminal_blocked(p)
                if t_blk:
                    continue
                pfm = part_fixed_machine.get(p)
                if pfm is not None and pfm != m:
                    continue
                if pfm is None and m in machine_fixed_parts:
                    continue

                # FIX: clear two-branch HZ single-machine guard
                if part_is_already_assigned(p, plan):
                    already_on_this = any(r["Machine"] == m for r in plan if r["Part"] == p)
                    if not already_on_this:
                        continue  # on a different machine — blocked
                    continue      # already on this machine — handled by extend above

                if not machine_has_capacity_for_new_part(m, p, plan):
                    continue
                daily_p = effective_daily(p)
                inv_now = current_inventory.get(p, 0)
                cap_q   = effective_opd_cap_qty(p, scenario_id, inv_now)
                if daily_p > 0 and inv_now >= cap_q:
                    continue
                unplanned.append(p)

            def _sort_key(p):
                inv_now_p = current_inventory.get(p, 0)
                dem_d     = demand_daily_raw.get(p, 0.0)
                dem_gap   = max(0.0, dem_d - inv_now_p) if dem_d > 0 else 0.0
                daily_p   = effective_daily(p)
                days_cov  = inv_now_p / daily_p if daily_p > 0 else 999
                cat_pri   = {"Runner": 0, "Repeater": 1, "Stranger": 2}.get(
                    part_category.get(p, "Stranger"), 2)
                needs_co  = 0 if (last_on_m is None or last_on_m == p) else 1
                p_col     = part_color.get(p, "UNKNOWN")
                same_col  = 0 if (needs_co == 1 and p_col == last_color and p_col != "UNKNOWN") else 1
                no_demand_gap = 0 if dem_gap > 0 else 1
                return (no_demand_gap, needs_co, same_col, cat_pri, -dem_gap, days_cov)

            unplanned.sort(key=_sort_key)

            for p in unplanned:
                if remaining < MIN_RUN_HOURS:
                    break
                if part_is_already_assigned(p, plan):
                    already_on_this = any(r["Machine"] == m for r in plan if r["Part"] == p)
                    if not already_on_this:
                        continue
                if not machine_has_capacity_for_new_part(m, p, plan):
                    continue
                co_hrs   = _co_hrs_for(p, m, machine_last_part)
                eff_free = round(remaining - co_hrs, 4)
                if eff_free < MIN_RUN_HOURS:
                    continue
                if co_hrs > 0 and _current_co_count(plan) >= MAX_DAILY_CO:
                    continue
                daily_p   = effective_daily(p)
                r_val     = rate.get(p, 1)
                inv_now_p = current_inventory.get(p, 0)
                cap_qty   = effective_opd_cap_qty(p, scenario_id, inv_now_p)
                headroom  = max(0.0, cap_qty - inv_now_p)
                if headroom <= 0:
                    continue
                shortfall = max(0.0, daily_p - inv_now_p)
                min_run   = max(MIN_RUN_HOURS, shortfall / r_val if r_val > 0 else MIN_RUN_HOURS)
                run_hrs   = max(min_run, min(eff_free, headroom / r_val if r_val > 0 else eff_free))
                qty       = round(run_hrs * r_val, 0)
                inv_before_p = inventory.get(p, 0)
                dem_d_p = demand_daily_raw.get(p, 0.0)
                _, t_note_p, t_relaxed_p = terminal_blocked(p)
                machine_hours[m]      = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
                current_inventory[p]  = round(current_inventory.get(p, 0) + qty, 0)
                machine_last_part[m]  = p
                already_planned.add(p)
                remaining = round(remaining - co_hrs - run_hrs, 4)
                last_on_m  = p
                last_color = part_color.get(p, "UNKNOWN")
                plan.append({
                    "Part": p, "Color": part_color.get(p, "UNKNOWN"),
                    "Category": part_category.get(p, "Stranger"),
                    "Fixed_Machine": part_fixed_machine.get(p, "—"),
                    "Fixed_Used": "N/A — filler", "Machine": m,
                    "Run_Hours": round(run_hrs, 3), "Changeover_Hrs": round(co_hrs, 3),
                    "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
                    "Rate_Per_Hour": round(r_val, 2),
                    "Production_Qty": qty,
                    "Inventory_Before": round(inv_before_p, 0),
                    "Demand_Today_Required": round(max(0.0, dem_d_p - inv_before_p), 2),
                    "Demand_Daily": round(dem_d_p, 2),
                    "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
                    "Effective_Daily": round(daily_p, 2),
                    "Demand_Driver": demand_driver(p),
                    "Monthly_Indent": round(effective_monthly(p), 0),
                    "Today_Target": round(today_target_qty.get(p, 0), 0),
                    "Changeover": "No" if co_hrs == 0 else "Yes",
                    "Color_Purge": "No",
                    "Terminal_Relaxed": "YES" if t_relaxed_p else "No",
                    "Terminal_Note": t_note_p if t_relaxed_p else "—",
                    "Terminal_Reminder": "—",
                    "Type": "Filler",
                    "Role": "Primary",
                    "HZ_Single_Machine": "ENFORCED",
                    "Runner_Lock": "No",
                    "Priority_Score": round(priority_scores.get(p, 0), 2),
                    "Phase": 2, "Stagger_Adjusted": "No",
                    "Indent_Met": "YES" if (inv_before_p + qty) >= (daily_p - 0.5) else "NO",
                    "Demand_Met": "YES" if is_demand_met(p, inv_before_p, qty)
                                  else ("N/A" if dem_d_p == 0 else "NO"),
                })
                print(f"    [FILL] {p:28s} -> {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}  driver={demand_driver(p)}")

        # Fill remaining with OPD/strategic extensions
        for ceiling in [opd_cap(scenario_id), STRATEGIC_BUFFER_DAYS, ABSOLUTE_MAX_DAYS]:
            remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
            if remaining < 0.001:
                break
            _, remaining = _extend_existing_on_machine(
                m, plan, machine_hours, current_inventory,
                priority_scores, ceiling, remaining)

        final_remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if final_remaining >= 0.25:
            util_final = round((1 - final_remaining / AVAILABLE_HOURS) * 100, 1)
            micro_idle_log.append({
                "Machine": m, "Idle_Hrs": round(final_remaining, 3),
                "Utilization_Pct": util_final,
                "Note": (f"Parts={parts_on_machine(m, plan)}/{MAX_PARTS_PER_MACHINE}  "
                         f"Exhausted compatible parts"),
            })

    return micro_idle_log

# =============================================================
# SECTION 20 — STRATEGIC BUFFER FILLER
# =============================================================

def strategic_buffer_score(part, current_inventory):
    daily = effective_daily(part)
    inv   = current_inventory.get(part, 0.0)
    r_val = rate.get(part, 1.0)
    cat   = part_category.get(part, "Stranger")
    if daily <= 0 or r_val <= 0:
        return 0.0
    velocity = daily / max(float(inv), 1.0)
    cat_w = {"Runner": 1.0, "Repeater": 0.7, "Stranger": 0.4}.get(cat, 0.4)
    return round(velocity * cat_w * STRATEGIC_PRIORITY_DISCOUNT * 100, 2)


def strategic_buffer_filler(plan, machine_hours, machine_last_part,
                              all_parts, already_planned,
                              current_inventory, scenario_id):
    print("\n" + "─"*65)
    print(f"  STRATEGIC BUFFER FILLER (HZ)")
    print("─"*65)
    filled_count = 0
    strat_scores = {p: strategic_buffer_score(p, current_inventory) for p in all_parts}
    machines_by_util = sorted(hz_machines, key=lambda m: machine_hours.get(m, 0))

    for m in machines_by_util:
        if m in _phase_a_machines:
            continue
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining < 0.001:
            continue

        for ceiling_days in [STRATEGIC_BUFFER_DAYS, ABSOLUTE_MAX_DAYS]:
            if remaining < 0.001:
                break
            consumed, remaining = _extend_existing_on_machine(
                m, plan, machine_hours, current_inventory,
                strat_scores, ceiling_days, remaining)
            if consumed > 0:
                filled_count += 1
            remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)

        if remaining < MIN_RUN_HOURS:
            continue

        if _current_co_count(plan) < MAX_DAILY_CO:
            last_on_m  = machine_last_part.get(m)
            last_color = part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"
            machine_compat = machine_compatible_parts.get(m, [])
            candidates = []
            for p in machine_compat:
                t_blk, _, _ = terminal_blocked(p)
                if t_blk:
                    continue
                if rate.get(p, 0) <= 0:
                    continue
                if indent_monthly.get(p, 0) <= 0 and demand_daily_raw.get(p, 0) <= 0:
                    continue
                pfm = part_fixed_machine.get(p)
                if pfm is not None and pfm != m:
                    continue
                if pfm is None and m in machine_fixed_parts:
                    continue
                if part_is_already_assigned(p, plan):
                    continue
                if not machine_has_capacity_for_new_part(m, p, plan):
                    continue
                daily_p = effective_daily(p)
                inv_p   = current_inventory.get(p, 0)
                if daily_p > 0 and inv_p >= ABSOLUTE_MAX_DAYS * daily_p:
                    continue
                candidates.append(p)

            def _strat_sort(p):
                needs_co = 0 if (last_on_m is None or last_on_m == p) else 1
                p_col    = part_color.get(p, "UNKNOWN")
                same_col = 0 if (needs_co == 1 and p_col == last_color and p_col != "UNKNOWN") else 1
                sc       = strat_scores.get(p, 0)
                return (needs_co, same_col, -sc)

            candidates.sort(key=_strat_sort)

            for p in candidates:
                if remaining < MIN_RUN_HOURS:
                    break
                if not machine_has_capacity_for_new_part(m, p, plan):
                    continue
                co_hrs   = _co_hrs_for(p, m, machine_last_part)
                eff_free = round(remaining - co_hrs, 4)
                if eff_free < MIN_RUN_HOURS:
                    continue
                if co_hrs > 0 and _current_co_count(plan) >= MAX_DAILY_CO:
                    continue
                r_val     = rate.get(p, 1)
                daily_p   = effective_daily(p)
                inv_now   = current_inventory.get(p, 0)
                headroom  = max(0.0, STRATEGIC_BUFFER_DAYS * daily_p - inv_now)
                if headroom <= 0:
                    headroom = max(0.0, ABSOLUTE_MAX_DAYS * daily_p - inv_now)
                if headroom <= 0:
                    continue
                run_hrs   = max(MIN_RUN_HOURS, min(eff_free, headroom / r_val if r_val > 0 else eff_free))
                qty       = round(run_hrs * r_val, 0)
                inv_before_p = inventory.get(p, 0)
                dem_d_p = demand_daily_raw.get(p, 0.0)
                has_purge = co_hrs > hz_changeover.get(m, DEFAULT_CHANGEOVER_HRS) + 0.001
                _, t_note_sb, t_relaxed_sb = terminal_blocked(p)
                machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
                current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
                machine_last_part[m] = p
                already_planned.add(p)
                remaining = round(remaining - co_hrs - run_hrs, 4)
                filled_count += 1
                plan.append({
                    "Part": p, "Color": part_color.get(p, "UNKNOWN"),
                    "Category": part_category.get(p, "?"),
                    "Fixed_Machine": part_fixed_machine.get(p, "—"),
                    "Fixed_Used": "N/A — strategic buffer", "Machine": m,
                    "Run_Hours": round(run_hrs, 3), "Changeover_Hrs": round(co_hrs, 3),
                    "Total_Hrs_Used": round(co_hrs + run_hrs, 3),
                    "Rate_Per_Hour": round(r_val, 2), "Production_Qty": qty,
                    "Inventory_Before": round(inv_before_p, 0),
                    "Demand_Today_Required": round(max(0.0, dem_d_p - inv_before_p), 2),
                    "Demand_Daily": round(dem_d_p, 2),
                    "Indent_Daily": round(indent_daily.get(p, 0.0), 2),
                    "Effective_Daily": round(daily_p, 2),
                    "Demand_Driver": demand_driver(p),
                    "Monthly_Indent": round(effective_monthly(p), 0),
                    "Today_Target": 0,
                    "Changeover": "No" if co_hrs == 0 else "Yes",
                    "Color_Purge": "Yes" if has_purge else "No",
                    "Terminal_Relaxed": "YES" if t_relaxed_sb else "No",
                    "Terminal_Note": t_note_sb if t_relaxed_sb else "—",
                    "Terminal_Reminder": "—",
                    "Type": "Strategic-Buffer",
                    "Role": "Buffer-Fill",
                    "HZ_Single_Machine": "ENFORCED",
                    "Runner_Lock": "No",
                    "Priority_Score": round(strat_scores.get(p, 0), 2),
                    "Phase": 4, "Stagger_Adjusted": "No",
                    "Indent_Met": "YES" if (inv_before_p + qty) >= (daily_p - 0.5) else "NO",
                    "Demand_Met": "YES" if is_demand_met(p, inv_before_p, qty)
                                  else ("N/A" if dem_d_p == 0 else "NO"),
                })
                last_on_m  = p
                last_color = part_color.get(p, "UNKNOWN")
                print(f"    [SB-NEW] {p:28s} -> {m:15s}  {run_hrs:.2f}h  qty={qty:.0f}")

    print(f"\n  Strategic buffer complete: {filled_count} extensions/additions")
    return filled_count

# =============================================================
# SECTION 21 — VALIDATION
# =============================================================

def validate_plan_rows(plan, current_inventory):
    violations = []
    part_machine_map = defaultdict(set)
    for row in plan:
        part_machine_map[row["Part"]].add(row["Machine"])

    for row in plan:
        p     = row["Part"]
        run_h = float(row.get("Run_Hours", 0) or 0)
        v = []
        if run_h < MIN_RUN_HOURS - 0.001:
            v.append(f"Run_Hours={run_h:.3f} < MIN={MIN_RUN_HOURS}")
        if len(part_machine_map[p]) > 1:
            v.append(f"HZ VIOLATION: part on {len(part_machine_map[p])} machines: "
                     f"{', '.join(part_machine_map[p])}")
        if v:
            violations.append({
                "Part": p, "Machine": row.get("Machine", "—"),
                "Run_Hours": run_h,
                "Demand_Daily": demand_daily_raw.get(p, 0.0),
                "Violations": " | ".join(v),
            })
    if violations:
        print(f"\n  VALIDATION: {len(violations)} violation(s)")
    else:
        print(f"\n  Validation: all rows OK (HZ single-machine rule verified)")
    return violations

# =============================================================
# SECTION 22 — CO STAGGER
# =============================================================

MIN_CO_GAP_HRS = 20 / 60.0

def _collect_co_events(plan, machines):
    events = []
    for m in machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if len(m_rows) < 2:
            continue
        cursor = 0.0
        for i, row in enumerate(m_rows):
            co_h  = float(row.get("Changeover_Hrs") or 0.0)
            run_h = float(row.get("Run_Hours") or 0.0)
            if co_h > 0 and i > 0:
                events.append({
                    "machine": m, "part_before": m_rows[i-1]["Part"],
                    "part_after": row["Part"], "co_duration": co_h,
                    "natural_start": cursor, "row_before": m_rows[i-1],
                    "row_after": row,
                })
            cursor += co_h + run_h
    return events

def _recompute_natural_start(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    cursor = 0.0
    for r in [r for r in plan if r["Machine"] == m]:
        if r is target:
            break
        cursor += float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
    return cursor

def _machine_spare(m, plan):
    used = sum(float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0) for r in plan if r["Machine"] == m)
    return max(0.0, AVAILABLE_HOURS - used)

def _extend_row_before(ev, wait_hrs, plan, machine_hours):
    m     = ev["machine"]
    spare = _machine_spare(m, plan)
    ext   = min(wait_hrs, spare)
    if ext <= 0:
        return 0.0, 0
    rb    = ev["row_before"]
    r_val = rate.get(rb["Part"], 1.0)
    extra = round(ext * r_val, 0)
    rb["Run_Hours"]      = round(float(rb.get("Run_Hours") or 0) + ext, 3)
    rb["Production_Qty"] = round(float(rb.get("Production_Qty") or 0) + extra, 0)
    rb["Total_Hrs_Used"] = round(float(rb.get("Changeover_Hrs") or 0) + float(rb["Run_Hours"]), 3)
    rb["Stagger_Adjusted"] = f"CO wait +{round(ext*60,1)}min"
    machine_hours[m] = round(machine_hours.get(m, 0) + ext, 4)
    return ext, extra

def stagger_changeovers(plan, machines, machine_hours):
    print(f"\n  Tool-Changer Serial Queue Scheduler (HZ — max {MAX_DAILY_CO} CO)")
    events = _collect_co_events(plan, machines)
    if not events:
        print(f"  No changeovers — tool changer idle")
        return
    events.sort(key=lambda e: e["natural_start"])
    if len(events) > MAX_DAILY_CO:
        events = events[:MAX_DAILY_CO]
    tool_changer_free_at = 0.0
    total_extra_pcs = 0
    for idx, ev in enumerate(events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        wait_hrs      = round(actual_start - natural_start, 4)
        tool_changer_free_at = actual_start + co_h
        if wait_hrs > 0.001:
            _, extra_pcs = _extend_row_before(ev, wait_hrs, plan, machine_hours)
            total_extra_pcs += extra_pcs
    print(f"  {len(events)} CO events processed  |  Extra pcs from wait fill: {total_extra_pcs}")

# =============================================================
# SECTION 23 — OUTPUT SHEET BUILDERS
# =============================================================

def build_production_vs_indent(plan, all_parts):
    if not plan:
        return pd.DataFrame()
    part_qty      = defaultdict(float)
    part_machines = defaultdict(list)
    for row in plan:
        p = row["Part"]
        part_qty[p]      += float(row.get("Production_Qty", 0))
        part_machines[p].append(row["Machine"])
    rows = []
    for p in sorted(part_qty.keys()):
        daily    = effective_daily(p)
        ind_d    = indent_daily.get(p, 0.0)
        dem_d    = demand_daily_raw.get(p, 0.0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_qty[p], 0)
        inv_after = round(inv_b + produced, 0)
        gap = round(produced - daily, 0)
        gap_dir = "OVER" if gap > 0 else ("UNDER" if gap < 0 else "MET")
        demand_met = is_demand_met(p, inv_b, produced)
        indent_met = (inv_b + produced) >= (ind_d - 0.5) if ind_d > 0 else True
        _, t_note, t_relaxed = terminal_blocked(p)
        rows.append({
            "Part": p, "Category": part_category.get(p, "Stranger"),
            "Machine": ", ".join(dict.fromkeys(part_machines[p])),
            "HZ_Single_Machine_Verified": "YES" if len(set(part_machines[p])) == 1 else "MULTI-MACHINE VIOLATION",
            "Total_Qty_Produced": produced,
            "Indent_Daily": round(ind_d, 2),
            "Demand_Daily": round(dem_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(p),
            "Terminal_Coverage_Ratio": terminal_coverage_ratio(p),
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Gap_vs_Effective_Daily": gap, "Gap_Direction": gap_dir,
            "Inventory_Before": round(inv_b, 0), "Inventory_After": inv_after,
            "Days_Coverage_After": round(inv_after / daily, 2) if daily > 0 else 0,
            "Demand_Met": "YES" if demand_met else ("N/A" if dem_d == 0 else "NO"),
            "Indent_Met": "YES" if indent_met else "NO",
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        order_map = {"UNDER": 0, "MET": 1, "OVER": 2}
        df["_sort"] = df["Gap_Direction"].map(order_map)
        df = df.sort_values(["_sort", "Gap_vs_Effective_Daily"]).drop(columns=["_sort"]).reset_index(drop=True)
    return df


def build_inventory_target_sheet(plan, all_parts, scenario_id):
    part_produced = defaultdict(float)
    for row in plan:
        part_produced[row["Part"]] += float(row.get("Production_Qty", 0))
    rows = []
    for p in sorted(all_parts):
        daily    = effective_daily(p)
        ind_d    = indent_daily.get(p, 0.0)
        dem_d    = demand_daily_raw.get(p, 0.0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_produced.get(p, 0), 0)
        inv_after = round(inv_b + produced, 0)
        days_after = round(inv_after / daily, 2) if daily > 0 else 0
        target_qty = round(TARGET_DAYS * daily, 0)
        if inv_after == 0:
            status = "CRITICAL"
        elif days_after < SAFETY_DAYS:
            status = "BELOW_SAFETY"
        elif days_after < TARGET_DAYS:
            status = "BUILDING"
        else:
            status = "AT_TARGET"
        _, t_note, t_relaxed = terminal_blocked(p)
        demand_met = is_demand_met(p, inv_b, produced)
        rows.append({
            "Part": p, "Category": part_category.get(p, "Stranger"),
            "Indent_Daily": round(ind_d, 2),
            "Demand_Daily": round(dem_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(p),
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Inv_Before": round(inv_b, 0),
            "Produced_Today": produced, "Inv_After": inv_after,
            "Days_Coverage_After": days_after, "Target_Qty_5days": target_qty,
            "Buffer_Status": status,
            "Demand_Met": "YES" if demand_met else ("N/A" if dem_d == 0 else "NO"),
            "Scheduled_Today": "YES" if produced > 0 else "NO",
        })
    df = pd.DataFrame(rows)
    if not df.empty:
        status_order = {"CRITICAL": 0, "BELOW_SAFETY": 1, "BUILDING": 2, "AT_TARGET": 3}
        df["_sort"] = df["Buffer_Status"].map(status_order)
        df = df.sort_values(["_sort"], ascending=True).drop(columns=["_sort"]).reset_index(drop=True)
    return df


def build_forward_look(current_inventory_after, all_parts):
    rows = []
    for p in all_parts:
        daily = effective_daily(p)
        if daily <= 0:
            continue
        inv_now = current_inventory_after.get(p, 0)
        days_now = inv_now / daily
        days_until_safety = max(0.0, round((inv_now - SAFETY_DAYS * daily) / daily, 1))
        days_until_zero   = max(0.0, round(inv_now / daily, 1))
        alert = ""
        if days_until_zero <= FORWARD_LOOK_DAYS:
            alert = f"ZERO-STOCK RISK in {days_until_zero:.1f} days"
        elif days_until_safety <= FORWARD_LOOK_DAYS:
            alert = f"BELOW SAFETY in {days_until_safety:.1f} days"
        if alert:
            rows.append({
                "Part": p, "Category": part_category.get(p, "Stranger"),
                "Demand_Daily": round(demand_daily_raw.get(p, 0.0), 2),
                "Effective_Daily": round(daily, 2),
                "Inv_After_Today": round(inv_now, 0),
                "Days_Coverage_Today": round(days_now, 2),
                "Days_Until_Safety": days_until_safety,
                "Days_Until_Zero": days_until_zero,
                "Alert": alert,
                "Action": "ESCALATE" if days_until_zero <= 2 else "Plan next 1-3 days",
            })
    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values("Days_Until_Zero").reset_index(drop=True)
    return df


def build_terminal_status_sheet():
    rows = []
    all_terminals_known = set(terminal_status.keys())
    for terms in part_terminals.values():
        for t in terms:
            all_terminals_known.add(t)
    for t in sorted(all_terminals_known):
        inv = terminal_status.get(t, None)
        parts_needing = [p for p, terms in part_terminals.items() if t in terms]
        parts_hard_blocked = []
        parts_relaxed      = []
        for p in parts_needing:
            dem_d = demand_daily_raw.get(p, 0.0)
            ind_d = indent_daily.get(p, 0.0)
            req_qty = max(dem_d, ind_d)
            r_val   = rate.get(p, 0.0)
            min_run_qty = MIN_RUN_HOURS * r_val if r_val > 0 else 0.0
            effective_req = max(req_qty, min_run_qty)
            hard_floor    = effective_req * (1.0 - TERMINAL_RELAXATION_PCT)
            t_inv = inv if inv is not None else 0
            if t_inv < hard_floor:
                parts_hard_blocked.append(f"{p}(need={effective_req:.0f},floor={hard_floor:.0f},inv={t_inv:.0f})")
            elif t_inv < effective_req:
                parts_relaxed.append(f"{p}(need={effective_req:.0f},inv={t_inv:.0f})")
        rows.append({
            "Terminal": t,
            "Inventory": round(inv, 0) if inv is not None else "NOT IN SHEET",
            "Parts_Requiring": ", ".join(sorted(parts_needing)) if parts_needing else "—",
            "Parts_Hard_Blocked": ", ".join(parts_hard_blocked) if parts_hard_blocked else "—",
            "Parts_Relaxed": ", ".join(parts_relaxed) if parts_relaxed else "—",
            "Impact": (
                "HARD BLOCKING" if parts_hard_blocked else
                ("RELAXED" if parts_relaxed else
                 ("Adequate" if parts_needing else "No parts"))
            ),
        })
    return pd.DataFrame(rows)


def build_machine_wise_plan(plan_df, machine_hours):
    if plan_df.empty:
        return pd.DataFrame()
    rows = []
    for m in hz_machines:
        machine_rows = plan_df[plan_df["Machine"] == m].copy()
        if machine_rows.empty:
            continue
        for _, pr in machine_rows.iterrows():
            p = pr.get("Part", "—")
            rows.append({
                "Machine": m, "Part": p,
                "Color": part_color.get(p, "UNKNOWN"),
                "Category": part_category.get(p, "Stranger"),
                "Role": pr.get("Role", "Primary"),
                "Phase": pr.get("Phase", 1),
                "Run_Hours": round(float(pr.get("Run_Hours", 0) or 0), 2),
                "Changeover_Hrs": round(float(pr.get("Changeover_Hrs", 0) or 0), 3),
                "Production_Qty": round(float(pr.get("Production_Qty", 0) or 0), 0),
                "Rate_Per_Hour": round(float(pr.get("Rate_Per_Hour", 0) or 0), 2),
                "Inventory_Before": round(float(pr.get("Inventory_Before", 0) or 0), 0),
                "Demand_Daily": round(float(pr.get("Demand_Daily", 0) or 0), 2),
                "Demand_Today_Required": round(float(pr.get("Demand_Today_Required", 0) or 0), 2),
                "Demand_Met": pr.get("Demand_Met", "—"),
                "Effective_Daily": round(float(pr.get("Effective_Daily", 0) or 0), 2),
                "Demand_Driver": pr.get("Demand_Driver", "—"),
                "Terminal_Relaxed": pr.get("Terminal_Relaxed", "No"),
                "Type": pr.get("Type", "Primary") or "Primary",
                "HZ_Single_Machine": pr.get("HZ_Single_Machine", "ENFORCED"),
                "Row_Type": "Part",
            })
        co_total  = machine_rows["Changeover_Hrs"].apply(lambda x: float(x) if pd.notna(x) else 0).sum()
        run_total = machine_rows["Run_Hours"].apply(lambda x: float(x) if pd.notna(x) else 0).sum()
        qty_total = machine_rows["Production_Qty"].apply(lambda x: float(x) if pd.notna(x) else 0).sum()
        hrs_total = round(co_total + run_total, 2)
        n_parts   = len(machine_rows["Part"].unique())
        rows.append({
            "Machine": m, "Part": f"TOTAL — {m}  [Parts={n_parts}/{MAX_PARTS_PER_MACHINE}]",
            "Color": "—", "Category": "—", "Role": "—", "Phase": "—",
            "Run_Hours": round(run_total, 2),
            "Changeover_Hrs": round(co_total, 2),
            "Production_Qty": round(qty_total, 0),
            "Rate_Per_Hour": "—", "Inventory_Before": "—",
            "Demand_Daily": "—", "Demand_Today_Required": "—", "Demand_Met": "—",
            "Effective_Daily": "—", "Demand_Driver": "—",
            "Terminal_Relaxed": "—",
            "Type": f"Total {hrs_total}h / {AVAILABLE_HOURS}h  |  Util {round(hrs_total/AVAILABLE_HOURS*100,1)}%",
            "HZ_Single_Machine": "—",
            "Row_Type": "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})
    return pd.DataFrame(rows)


def build_daily_totals_sheet(plan, machine_hours, hz_machines_list):
    if not plan:
        return pd.DataFrame()
    total_qty     = sum(float(r.get("Production_Qty", 0)) for r in plan)
    total_co      = sum(1 for r in plan if r.get("Changeover") == "Yes")
    total_co_hrs  = sum(float(r.get("Changeover_Hrs", 0)) for r in plan)
    total_run_hrs = sum(float(r.get("Run_Hours", 0)) for r in plan)
    parts_scheduled = len({r["Part"] for r in plan})
    parts_with_demand = [p for p in {r["Part"] for r in plan} if demand_daily_raw.get(p, 0) > 0]
    demand_met_count = sum(
        1 for p in parts_with_demand
        if is_demand_met(p, inventory.get(p, 0),
                         sum(float(r.get("Production_Qty", 0)) for r in plan if r["Part"] == p))
    )
    multi_machine_violations = []
    part_machine_map = defaultdict(set)
    for r in plan:
        part_machine_map[r["Part"]].add(r["Machine"])
    for p, machines_set in part_machine_map.items():
        if len(machines_set) > 1:
            multi_machine_violations.append(f"{p}({', '.join(machines_set)})")

    rows = [
        {"Metric": "Planning Date",                                   "Value": str(PLANNING_DATE)},
        {"Metric": "Scheduler Version",                               "Value": "HZ V1"},
        {"Metric": "HZ Rule: One Machine Per Part",                   "Value": "ENFORCED"},
        {"Metric": "Total Production Qty",                            "Value": round(total_qty, 0)},
        {"Metric": "Total Changeovers Today",                         "Value": total_co},
        {"Metric": "Max Changeovers Allowed (HZ)",                    "Value": MAX_DAILY_CO},
        {"Metric": "Total Changeover Hours",                          "Value": round(total_co_hrs, 2)},
        {"Metric": "Total Run Hours",                                  "Value": round(total_run_hrs, 2)},
        {"Metric": "Parts Scheduled",                                  "Value": parts_scheduled},
        {"Metric": "Parts With Demand",                               "Value": len(parts_with_demand)},
        {"Metric": "Demand Met (production + stock)",                 "Value": demand_met_count},
        {"Metric": "Demand NOT Met",                                  "Value": len(parts_with_demand) - demand_met_count},
        {"Metric": "Multi-Machine Violations (should be 0)",          "Value": len(multi_machine_violations)},
        {"Metric": "Violation Detail",
         "Value": "; ".join(multi_machine_violations) if multi_machine_violations else "None"},
        {"Metric": "",                                                 "Value": ""},
        {"Metric": "=== MACHINE-WISE TOTALS ===",                     "Value": ""},
    ]
    for m in sorted(hz_machines_list):
        m_rows = [r for r in plan if r["Machine"] == m]
        if not m_rows:
            rows.append({"Metric": f"{m} — IDLE", "Value": 0})
            continue
        m_qty   = sum(float(r.get("Production_Qty", 0)) for r in m_rows)
        m_co    = sum(1 for r in m_rows if r.get("Changeover") == "Yes")
        m_hrs   = machine_hours.get(m, 0)
        m_util  = round(m_hrs / AVAILABLE_HOURS * 100, 1)
        m_parts = len({r["Part"] for r in m_rows})
        rows.append({
            "Metric": f"{m} — Parts={m_parts}/{MAX_PARTS_PER_MACHINE} | CO={m_co} | Util={m_util}% | Hrs={round(m_hrs,2)}",
            "Value": round(m_qty, 0)
        })
    return pd.DataFrame(rows)


def build_planning_summary_sheet(plan, not_planned_list, deferred_list,
                                  all_parts, machine_hours, hz_machines_list,
                                  already_planned_set, scenario_desc):
    rows = []
    def _sep(label=""):
        rows.append({"Section": f"── {label} ──", "Metric": "", "Count_or_Value": "", "Detail": ""})
    def _row(section, metric, value, detail=""):
        rows.append({"Section": section, "Metric": metric, "Count_or_Value": value, "Detail": detail})

    part_produced = defaultdict(float)
    for r in plan:
        part_produced[r["Part"]] += float(r.get("Production_Qty", 0))

    _sep("SCENARIO")
    _row("Scenario", "Active Scenario", scenario_desc)
    _row("HZ Rules", "Single Machine Per Part", "ENFORCED",
         "A part can NEVER run on 2 machines in HZ section")
    _row("HZ Rules", "Max Changeovers", MAX_DAILY_CO,
         "45 COs allowed per day in HZ (vs 25 in VT)")

    _sep("PLANNING COUNTS")
    _row("Planning Counts", "Total Parts in Universe",    len(all_parts))
    _row("Planning Counts", "Parts Successfully Planned", len({r["Part"] for r in plan}))
    _row("Planning Counts", "Parts NOT Planned",          len(not_planned_list))
    _row("Planning Counts", "Parts Deferred",             len(deferred_list))

    _sep("DEMAND COVERAGE")
    demand_parts = [p for p in all_parts if demand_daily_raw.get(p, 0) > 0]
    dem_stock = dem_produced = dem_not = 0
    dem_fail_list = []
    for p in demand_parts:
        inv_b    = inventory.get(p, 0)
        produced = part_produced.get(p, 0)
        dem_d    = demand_daily_raw.get(p, 0)
        if inv_b >= dem_d:
            dem_stock += 1
        elif is_demand_met(p, inv_b, produced):
            dem_produced += 1
        else:
            dem_not += 1
            dem_fail_list.append(p)
    _row("Demand Coverage", "Parts with Demand",               len(demand_parts))
    _row("Demand Coverage", "Met — by existing stock alone",   dem_stock)
    _row("Demand Coverage", "Met — by production today",       dem_produced)
    _row("Demand Coverage", "NOT Met (shortfall remains)",     dem_not,
         ", ".join(dem_fail_list) if dem_fail_list else "—")

    _sep("MACHINE UTILISATION")
    util_list = []
    for m in sorted(hz_machines_list):
        used     = machine_hours.get(m, 0)
        util_pct = round(used / AVAILABLE_HOURS * 100, 1)
        util_list.append(util_pct)
        parts_on = len({r["Part"] for r in plan if r["Machine"] == m})
        co_on    = sum(1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes")
        status   = "FULL" if used >= AVAILABLE_HOURS - 0.3 else "GOOD" if util_pct >= 98 else "OK" if util_pct >= 90 else "UNDERUSED"
        _row("Machine Util", m, f"{util_pct}%",
             f"Hours={round(used,2)}/{AVAILABLE_HOURS}  Parts={parts_on}/{MAX_PARTS_PER_MACHINE}  CO={co_on}  [{status}]")
    if util_list:
        _row("Machine Util", "AVERAGE UTILISATION", f"{round(sum(util_list)/len(util_list),1)}%")

    _sep("NOT PLANNED — REASONS")
    if not_planned_list:
        for r in not_planned_list:
            _row("Not Planned", r.get("Part", "?"), "NOT PLANNED", r.get("Reason", "—"))
    else:
        _row("Not Planned", "—", "All eligible parts planned", "")

    return pd.DataFrame(rows)


def build_part_decision_sheet(all_parts, plan, not_planned_list, deferred_list, priority_scores):
    not_planned_reasons = {r["Part"]: r["Reason"] for r in not_planned_list if "Part" in r}
    deferred_reasons    = {r["Part"]: r["Reason"] for r in deferred_list if "Part" in r}

    part_produced = defaultdict(float)
    part_machines = defaultdict(list)
    for row in plan:
        part_produced[row["Part"]] += float(row.get("Production_Qty", 0))
        part_machines[row["Part"]].append(row["Machine"])

    rows = []
    for p in sorted(all_parts):
        inv_b    = inventory.get(p, 0)
        dem_d    = demand_daily_raw.get(p, 0.0)
        ind_d    = indent_daily.get(p, 0.0)
        daily    = effective_daily(p)
        r_val    = rate.get(p, None)
        produced = round(part_produced.get(p, 0), 0)
        inv_after = round(inv_b + produced, 0)
        machines  = ", ".join(dict.fromkeys(part_machines[p])) if p in part_machines else "—"
        dem_gap   = max(0.0, dem_d - inv_b)
        dem_met   = is_demand_met(p, inv_b, produced)
        td = get_terminal_detail_for_part(p)
        t_blocked = td["Terminal_Blocked"] == "YES"
        t_relaxed = td["Terminal_Relaxed"] == "YES"
        t_reason  = td["Terminal_Block_Reason"]
        in_plan = produced > 0

        machines_count = len(set(part_machines[p])) if p in part_machines else 0
        hz_rule_ok = machines_count <= 1

        if in_plan:
            plan_reason = f"PLANNED — dem={dem_d:.2f} inv_before={inv_b:.0f}"
            if not hz_rule_ok:
                plan_reason += " [HZ VIOLATION: MULTI-MACHINE]"
        elif p in not_planned_reasons:
            plan_reason = f"NOT PLANNED — {not_planned_reasons[p]}"
        elif p in deferred_reasons:
            plan_reason = f"DEFERRED — {deferred_reasons[p]}"
        elif t_blocked:
            plan_reason = f"BLOCKED — {t_reason}"
        elif r_val is None or r_val == 0:
            plan_reason = "SKIPPED — zero/missing cycle time"
        elif daily == 0:
            plan_reason = "SKIPPED — no demand and no indent"
        else:
            skip, skip_reason = should_skip(p)
            plan_reason = f"SKIPPED — {skip_reason}" if skip else "NOT SCHEDULED — no machine capacity"

        rows.append({
            "Part": p,
            "Category": part_category.get(p, "Stranger"),
            "Fixed_Machine": part_fixed_machine.get(p, "—"),
            "Priority_Score": round(priority_scores.get(p, 0), 2),
            "Inventory_Before": round(inv_b, 0),
            "Demand_Daily": round(dem_d, 2),
            "Indent_Daily": round(ind_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Gap_Today": round(dem_gap, 2),
            "Produced_Today": produced,
            "Inventory_After": inv_after,
            "Machine_Used": machines,
            "Machines_Count": machines_count,
            "HZ_Single_Machine_Rule_OK": "YES" if hz_rule_ok else "VIOLATION",
            "Demand_Met": "YES" if dem_met else ("N/A" if dem_d == 0 else "NO"),
            "Indent_Met": "YES" if (inv_after >= daily - 0.5 and daily > 0) else ("N/A" if daily == 0 else "NO"),
            "In_Plan": "YES" if in_plan else "NO",
            "Plan_Reason": plan_reason,
            "Terminal_Hard_Blocked": td.get("Terminal_Blocked", "No"),
            "Terminal_Relaxed": td.get("Terminal_Relaxed", "No"),
        })
    return pd.DataFrame(rows)


def build_audit(all_universe_parts, matrix_parts, zero_rate_set):
    audit_rows = []
    for part in all_universe_parts:
        inv    = inventory.get(part, 0.0)
        r_val  = rate.get(part, None)
        monthly = effective_monthly(part)
        ind_d  = indent_daily.get(part, 0.0)
        dem_d  = demand_daily_raw.get(part, 0.0)
        daily  = effective_daily(part)
        days_cov = inv / daily if daily > 0 else 0
        t_blk, t_rsn, t_relaxed = terminal_blocked(part)

        if part in zero_rate_set or r_val is None:
            status = "ZERO/MISSING CYCLE TIME"
        elif part not in matrix_parts:
            status = "NOT IN HZ_MATRIX"
        elif monthly == 0:
            status = "ZERO INDENT+DEMAND"
        elif daily <= MIN_DAILY_INDENT and dem_d <= 0:
            status = "SKIPPED (LOW INDENT, NO DEMAND)"
        elif daily > 0 and inv >= TARGET_DAYS * daily:
            status = "AT TARGET — SKIP"
        elif t_blk:
            status = "BLOCKED — TERMINAL (HARD)"
        elif t_relaxed:
            status = "ENTERS SCHEDULER [TERMINAL RELAXED]"
        else:
            status = "ENTERS SCHEDULER"

        audit_rows.append({
            "Part": part, "Color": part_color.get(part, "UNKNOWN"),
            "Indent_Daily": round(ind_d, 2),
            "Demand_Daily": round(dem_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(part),
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Terminal_Note": t_rsn if (t_relaxed or t_blk) else "—",
            "Inventory": round(inv, 0), "Days_Coverage": round(days_cov, 2),
            "Rate_Per_Hour": round(r_val, 2) if r_val else "—",
            "Status": status,
            "HZ_Note": "Single machine per part — enforced throughout",
        })
    return pd.DataFrame(audit_rows)


def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv    = inventory.get(p, 0.0)
        monthly = effective_monthly(p)
        daily  = effective_daily(p)
        skip, skip_reason = should_skip(p)
        days_cov = inv / daily if daily > 0 else 0
        _, t_note, t_relaxed = terminal_blocked(p)
        if inv == 0.0 and monthly > 0:
            status = "ZERO INV"
        elif skip and inv >= TARGET_DAYS * daily:
            status = "AT TARGET"
        elif skip:
            status = "SKIPPED"
        else:
            status = "PRODUCTION NEEDED"
        rows.append({
            "Part": p,
            "Indent_Monthly": round(indent_monthly.get(p, 0.0), 0),
            "Demand_Daily": round(demand_daily_raw.get(p, 0.0), 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(p),
            "Terminal_Relaxed": "YES" if t_relaxed else "No",
            "Inventory_Now": round(inv, 0),
            "Days_Coverage": round(days_cov, 2),
            "Indent_Status": status,
            "Skip_Reason": skip_reason,
        })
    return pd.DataFrame(rows)


def build_planned_vs_unplanned_sheet(all_parts, plan, not_planned_list, deferred_list,
                                      terminal_blocked_parts):
    not_planned_set = {r["Part"] for r in not_planned_list if "Part" in r}
    not_planned_reasons = {r["Part"]: r["Reason"] for r in not_planned_list if "Part" in r}
    deferred_set    = {r["Part"] for r in deferred_list if "Part" in r}
    deferred_reasons = {r["Part"]: r["Reason"] for r in deferred_list if "Part" in r}
    terminal_block_set = {r["Part"] for r in terminal_blocked_parts if "Part" in r}
    terminal_block_reasons = {r["Part"]: r["Reason"] for r in terminal_blocked_parts if "Part" in r}

    part_produced = defaultdict(float)
    part_machines_planned = defaultdict(list)
    for row in plan:
        part_produced[row["Part"]] += float(row.get("Production_Qty", 0))
        part_machines_planned[row["Part"]].append(row["Machine"])

    rows = []
    for p in all_parts:
        inv_b   = inventory.get(p, 0.0)
        dem_d   = demand_daily_raw.get(p, 0.0)
        ind_d   = indent_daily.get(p, 0.0)
        daily   = effective_daily(p)
        r_val   = rate.get(p, None)
        produced = round(part_produced.get(p, 0), 0)
        inv_after = round(inv_b + produced, 0)
        days_before = round(inv_b / daily, 2) if daily > 0 else 0
        days_after  = round(inv_after / daily, 2) if daily > 0 else 0
        compat_machines = hz_compat.get(p, [])
        td = get_terminal_detail_for_part(p)
        t_blocked_flag = td["Terminal_Blocked"] == "YES"
        t_relaxed_flag = td["Terminal_Relaxed"] == "YES"
        is_planned  = produced > 0
        is_t_block  = p in terminal_block_set

        if is_planned:
            status = "PLANNED" + (" (Terminal Relaxed)" if t_relaxed_flag else "")
        elif is_t_block:
            status = "TERMINAL BLOCKED"
        elif p in not_planned_set:
            status = "NOT PLANNED"
        elif p in deferred_set:
            status = "DEFERRED"
        else:
            skip, skip_reason = should_skip(p)
            if skip:
                status = "SKIPPED"
            elif r_val is None or r_val == 0:
                status = "SKIPPED — no rate"
            elif daily == 0:
                status = "SKIPPED — no demand/indent"
            else:
                status = "NOT SCHEDULED"

        machines_used = list(dict.fromkeys(part_machines_planned[p]))
        hz_check = "OK" if len(machines_used) <= 1 else f"VIOLATION — {len(machines_used)} machines"

        rows.append({
            "Part": p,
            "Category": part_category.get(p, "Stranger"),
            "Status": status,
            "Status_Group": "PLANNED" if is_planned else "NOT PLANNED",
            "HZ_Single_Machine_Check": hz_check,
            "Demand_Daily": round(dem_d, 2),
            "Indent_Daily": round(ind_d, 2),
            "Effective_Daily": round(daily, 2),
            "Demand_Driver": demand_driver(p),
            "Inventory_Before": round(inv_b, 0),
            "Days_Coverage_Before": days_before,
            "Produced_Today": produced,
            "Inventory_After": inv_after,
            "Days_Coverage_After": days_after,
            "Compatible_Machines_Count": len(compat_machines),
            "Machine_Used": ", ".join(machines_used) if machines_used else "—",
            "Terminal_Blocked": td.get("Terminal_Blocked", "No"),
            "Terminal_Relaxed": td.get("Terminal_Relaxed", "No"),
        })

    df = pd.DataFrame(rows)
    if not df.empty:
        group_order = {"NOT PLANNED": 0, "PLANNED": 1}
        df["_group_sort"] = df["Status_Group"].map(group_order).fillna(0)
        df = df.sort_values(["_group_sort", "Demand_Daily"], ascending=[True, False])
        df = df.drop(columns=["_group_sort", "Status_Group"]).reset_index(drop=True)
    return df

# =============================================================
# SECTION 24 — MAIN SCHEDULER
# =============================================================

def get_terminal_eligible_parts(parts, exclude_fixed=True):
    fixed_set = set(part_fixed_machine.keys()) if exclude_fixed else set()
    eligible_non_fixed = []
    eligible_fixed     = []
    blocked_info       = []
    non_fixed_parts = [p for p in parts if p not in fixed_set]
    non_fixed_parts.sort(key=lambda p: (-demand_daily_raw.get(p, 0.0), -terminal_coverage_ratio(p)))
    for p in non_fixed_parts:
        blocked, reason, relaxed = terminal_blocked(p)
        if blocked:
            blocked_info.append({"Part": p, "Reason": reason, "Fixed": "No"})
        else:
            eligible_non_fixed.append(p)
    for p in [p for p in parts if p in fixed_set]:
        blocked, reason, relaxed = terminal_blocked(p)
        if blocked:
            blocked_info.append({"Part": p, "Reason": reason, "Fixed": "YES (fixed machine)"})
        else:
            eligible_fixed.append(p)
    return eligible_non_fixed, eligible_fixed, blocked_info


def schedule(parts, label=""):
    global _phase_a_machines, _dynamic_max_parts
    print("\n" + "─"*65)
    print(f"  {label}  |  {len(parts)} parts  |  {len(hz_machines)} machines")
    print(f"  HZ RULE: One machine per part — enforced throughout")
    print("─"*65)

    scenario_id, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")
    print(f"  OPD cap: {opd_cap(scenario_id)} days")

    horizon_df = compute_indent_horizon(parts)

    print(f"\n  TERMINAL ELIGIBILITY PRE-FILTER (HZ)")
    eligible_non_fixed, eligible_fixed, terminal_blocked_parts_info = get_terminal_eligible_parts(
        parts, exclude_fixed=True)

    if terminal_blocked_parts_info:
        print(f"\n  TERMINAL BLOCKED ({len(terminal_blocked_parts_info)} parts):")
        for info in terminal_blocked_parts_info:
            print(f"    X {info['Part']:<30}  {info['Reason'][:70]}")
    else:
        print(f"\n  No parts hard-blocked by terminals.")

    eligible_all = list(dict.fromkeys(eligible_non_fixed + eligible_fixed + list(part_fixed_machine.keys())))
    active_parts_eligible = [
        p for p in eligible_all
        if not should_skip(p)[0] and effective_daily(p) > 0
    ]

    priority_scores, score_rows = compute_priority_scores(active_parts_eligible)
    score_df = pd.DataFrame(score_rows) if score_rows else pd.DataFrame()

    machine_pre_assignment, _dynamic_max_parts = pre_distribute_parts_to_machines(
        active_parts_eligible, hz_machines
    )

    machine_hours         = {m: 0.0 for m in hz_machines}
    machine_last_part     = {m: machine_state.get(m) for m in hz_machines}
    current_inventory     = inventory.copy()
    inventory_start_of_day = inventory.copy()

    plan            = []
    already_planned = set()
    not_planned     = []
    deferred        = []

    # ── FIXED MACHINE PASS ─────────────────────────────────────
    if machine_fixed_parts:
        _phase_a_machines, _ = schedule_fixed_machines(
            machine_hours, machine_last_part, current_inventory,
            plan, already_planned, priority_scores, scenario_id
        )
    else:
        _phase_a_machines = set()

    # ── PASS 0: DEMAND-FIRST ───────────────────────────────────
    print("\n" + "─"*65)
    print(f"  PASS 0: DEMAND-FIRST (largest demand gap first)")
    print(f"  HZ: one machine per part, stops after first machine")
    print("─"*65)

    demand_parts_sorted = sorted(
        [p for p in eligible_non_fixed
         if demand_daily_raw.get(p, 0) > 0
         and current_inventory.get(p, 0) < demand_daily_raw.get(p, 0)
         and p not in already_planned],
        key=lambda p: (-demand_daily_raw.get(p, 0), -terminal_coverage_ratio(p))
    )

    for part in demand_parts_sorted:
        rows = assign_demand_for_part(
            part, scenario_id, machine_hours, machine_last_part,
            current_inventory, plan, already_planned, priority_scores
        )
        if rows:
            plan.extend(rows)
            print(f"    DEMAND-FIRST {part:28s}  gap={demand_daily_raw.get(part,0)-inventory.get(part,0):.0f}  "
                  f"qty={sum(float(r['Production_Qty']) for r in rows):.0f}")
        else:
            not_planned.append({
                "Part": part,
                "Reason": f"Demand gap exists but no machine capacity available"
            })

    # ── PASS 1: RUNNER PRIORITY ────────────────────────────────
    runner_log = enforce_runner_priority(
        plan, machine_hours, machine_last_part,
        current_inventory, already_planned,
        priority_scores, inventory_start_of_day
    )

    # ── PASS 2: INVENTORY BUILD ────────────────────────────────
    print("\n" + "─"*65)
    print(f"  PASS 2: INVENTORY BUILD (priority-sorted, HZ single-machine)")
    print("─"*65)

    sorted_active = sorted(
        active_parts_eligible,
        key=lambda p: priority_scores.get(p, 0),
        reverse=True
    )

    for part in sorted_active:
        if part in already_planned:
            assigned_m = next((r["Machine"] for r in plan if r["Part"] == part), None)
            if assigned_m:
                _extend_part_on_machine(
                    part, assigned_m, plan, scenario_id, machine_hours, current_inventory,
                    rate.get(part, 1), effective_daily(part), current_inventory.get(part, 0)
                )
            continue

        skip, skip_reason = should_skip(part)
        if skip:
            deferred.append({"Part": part, "Reason": skip_reason})
            continue

        if not hz_compat.get(part):
            not_planned.append({"Part": part, "Reason": "Not in HZ compatibility matrix"})
            continue

        new_rows = assign_inventory_build(
            part, scenario_id, machine_hours, machine_last_part,
            current_inventory, plan, already_planned, priority_scores
        )
        if not new_rows and part not in already_planned:
            not_planned.append({"Part": part, "Reason": "No machine capacity available"})

    # ── PASS 3: UTILIZATION ENFORCER ──────────────────────────
    micro_idle_log = utilization_enforcer(
        plan, machine_hours, machine_last_part,
        list(hz_compat.keys()), already_planned,
        current_inventory, scenario_id, priority_scores
    )

    # ── STRATEGIC BUFFER FILLER ───────────────────────────────
    strategic_buffer_filler(
        plan, machine_hours, machine_last_part,
        list(hz_compat.keys()), already_planned,
        current_inventory, scenario_id
    )

    # ── RESEQUENCE + RECONCILE ────────────────────────────────
    plan = resequence_machine_rows(plan, machine_state)
    reconcile_machine_hours(plan, machine_hours)

    # ── STAGGER CHANGEOVERS ───────────────────────────────────
    stagger_changeovers(plan, hz_machines, machine_hours)
    reconcile_machine_hours(plan, machine_hours)

    # ── VALIDATION ────────────────────────────────────────────
    violations = validate_plan_rows(plan, current_inventory)

    # ── UPDATE PRIORITY SCORES ────────────────────────────────
    for row in plan:
        if row.get("Priority_Score") == 0:
            row["Priority_Score"] = priority_scores.get(row["Part"], 0)

    # ── PRINT SUMMARY ─────────────────────────────────────────
    print("\n" + "─"*65)
    print(f"  HZ SCHEDULE COMPLETE")
    print(f"  Planned: {len({r['Part'] for r in plan})} parts")
    print(f"  Not planned: {len(not_planned)}")
    print(f"  Deferred: {len(deferred)}")
    total_co = sum(1 for r in plan if r.get('Changeover')=='Yes')
    print(f"  Changeovers: {total_co} / {MAX_DAILY_CO} allowed")
    print(f"  Violations: {len(violations)}")
    avg_util = round(
        sum(machine_hours.get(m, 0) for m in hz_machines) / len(hz_machines) / AVAILABLE_HOURS * 100, 1
    ) if hz_machines else 0
    print(f"  Avg machine utilization: {avg_util}%")

    part_machine_map = defaultdict(set)
    for r in plan:
        part_machine_map[r["Part"]].add(r["Machine"])
    multi_violations = [p for p, ms in part_machine_map.items() if len(ms) > 1]
    if multi_violations:
        print(f"  !! HZ RULE VIOLATIONS (multi-machine): {multi_violations}")
    else:
        print(f"  HZ Rule Check: PASSED — no part on more than 1 machine")
    print("─"*65)

    # ── BUILD OUTPUT SHEETS ───────────────────────────────────
    plan_df = pd.DataFrame(plan) if plan else pd.DataFrame()
    all_parts_universe = list(set(
        list(hz_compat.keys()) +
        list(indent_monthly.keys()) +
        list(demand_daily_raw.keys())
    ))
    matrix_parts_set = set(hz_compat.keys())
    zero_rate_set    = set(data_zero_rate["Material"].tolist())

    sheets = {
        "HZ_Plan":               plan_df,
        "HZ_Machine_Plan":       build_machine_wise_plan(plan_df, machine_hours),
        "HZ_Priority_Scores":    score_df,
        "HZ_Prod_vs_Indent":     build_production_vs_indent(plan, all_parts_universe),
        "HZ_Inventory_Target":   build_inventory_target_sheet(plan, all_parts_universe, scenario_id),
        "HZ_Forward_Look":       build_forward_look(current_inventory, all_parts_universe),
        "HZ_Part_Decision":      build_part_decision_sheet(
                                     all_parts_universe, plan, not_planned, deferred, priority_scores),
        "HZ_Planned_vs_Unplanned": build_planned_vs_unplanned_sheet(
                                     all_parts_universe, plan, not_planned, deferred,
                                     terminal_blocked_parts_info),
        "HZ_Terminal_Status":    build_terminal_status_sheet(),
        "HZ_Runner_Priority":    pd.DataFrame(runner_log) if runner_log else pd.DataFrame(),
        "HZ_CO_Queue":           build_co_queue(plan, hz_machines),
        "HZ_Micro_Idle":         pd.DataFrame(micro_idle_log) if micro_idle_log else pd.DataFrame(),
        "HZ_Audit":              build_audit(all_parts_universe, matrix_parts_set, zero_rate_set),
        "HZ_Daily_Totals":       build_daily_totals_sheet(plan, machine_hours, hz_machines),
        "HZ_Summary":            build_planning_summary_sheet(
                                     plan, not_planned, deferred,
                                     all_parts_universe, machine_hours, hz_machines,
                                     already_planned, scenario_desc),
        "HZ_Indent_Horizon":     horizon_df,
        "HZ_Violations":         pd.DataFrame(violations) if violations else pd.DataFrame(),
    }

    return sheets, machine_hours, machine_last_part, current_inventory, scenario_desc


def build_co_queue(plan, machines):
    events = _collect_co_events(plan, machines)
    if not events:
        return pd.DataFrame()
    events.sort(key=lambda e: e["natural_start"])
    rows = []
    tool_changer_free_at = 0.0
    for pos, ev in enumerate(events, 1):
        natural_start = _recompute_natural_start(ev, plan)
        co_h          = ev["co_duration"]
        actual_start  = max(natural_start, tool_changer_free_at)
        tool_changer_free_at = actual_start + co_h
        before_color = part_color.get(ev["part_before"], "UNKNOWN")
        after_color  = part_color.get(ev["part_after"],  "UNKNOWN")
        color_change = before_color != after_color and before_color != "UNKNOWN" and after_color != "UNKNOWN"
        rows.append({
            "Queue_Position": pos, "Machine": ev["machine"],
            "Part_Before": ev["part_before"], "Part_After": ev["part_after"],
            "Color_Before": before_color, "Color_After": after_color,
            "Color_Change": "YES — PURGE" if color_change else "No",
            "CO_Duration_Min": round(co_h * 60, 1),
        })
    return pd.DataFrame(rows)

# =============================================================
# SECTION 25 — RUN & WRITE OUTPUT
# =============================================================

print("\n" + "─"*65)
print(f"  RUNNING HZ SCHEDULER")
print(f"  Key rules: Single machine per part | Max {MAX_DAILY_CO} changeovers")
print("─"*65)

all_hz_parts = list(set(
    list(hz_compat.keys()) +
    list(indent_monthly.keys()) +
    list(demand_daily_raw.keys())
))

sheets, final_machine_hours, final_machine_last_part, final_inventory, scenario_desc = schedule(
    all_hz_parts, label="HZ SCHEDULER"
)

save_machine_state(final_machine_last_part)

print(f"\n  Writing output: {output_path}")
with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        if df is not None and not df.empty:
            df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
            print(f"    Sheet '{sheet_name[:31]}' written  ({len(df)} rows)")
        else:
            pd.DataFrame({"Note": [f"No data for {sheet_name}"]}).to_excel(
                writer, sheet_name=sheet_name[:31], index=False
            )

print("\n" + "─"*65)
print(f"  Smart APS HZ V1 COMPLETE")
print(f"  Output file  : {output_path}")
print(f"  Planning date: {PLANNING_DATE}")
print(f"  Scenario     : {scenario_desc}")
total_planned = len({r["Part"] for r in sheets["HZ_Plan"].to_dict("records")}) if not sheets["HZ_Plan"].empty else 0
print(f"  Parts planned: {total_planned}")
avg_util = round(
    sum(final_machine_hours.get(m, 0) for m in hz_machines) / len(hz_machines) / AVAILABLE_HOURS * 100, 1
) if hz_machines else 0
print(f"  Avg utilization: {avg_util}%")
print(f"  HZ Rule: One machine per part — enforced and validated")
print(f"{'='*65}\n")